<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/GES_Stage6C_Cell_6C_4F0_Gene_Level_Inference_Materialization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==================================================================================================
# COLAB NOTEBOOK FILE NAME:
# GES_Stage6C_Cell_6C_4F0_Gene_Level_Inference_Materialization.ipynb
#
# STAGE 6C STEP 4F — CELL 6C-4F0
# GENE-LEVEL POINT ESTIMATES, PAIRED BOOTSTRAP INFERENCE,
# PRIMARY-GENE HOLM CORRECTION, MATERIALIZATION, AND FREEZE
#
# Exact historical implementation reproduced:
#   - Four fixed gene strata: BRCA1, BRCA2, MLH1, exploratory EGFR
#   - 2,000 ordinary row-bootstrap attempts per gene
#   - One continuous np.random.default_rng(42) stream
#   - Fixed processing order: BRCA1, BRCA2, MLH1, EGFR
#   - Multinomial row-multiplicity representation
#   - Batch size 50
#   - Identical resamples across Full GES, No-star GES, Review stars,
#     and Combined metadata within each gene
#   - Exact tie-aware grouped weighted AUPRC and AUROC
#   - Full-GES-minus-comparator paired inference
#   - Holm correction separately across nine primary-gene AUPRC comparisons
#     and nine primary-gene AUROC comparisons
#   - EGFR excluded from primary multiplicity families
#
# Scientific boundary:
#   - No score, outcome, gene assignment, threshold, model, feature, weight,
#     linkage decision, row order, or cohort membership is changed.
#   - EGFR remains exploratory.
#   - Experiment 2 is not started.
# ==================================================================================================


# --------------------------------------------------------------------------------------------------
# 0. MOUNT GOOGLE DRIVE
# --------------------------------------------------------------------------------------------------

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False,
)


# --------------------------------------------------------------------------------------------------
# 1. IMPORTS
# --------------------------------------------------------------------------------------------------

from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path

import gc
import hashlib
import json
import os
import platform
import re
import sys
import time

import joblib
import numpy as np
import pandas as pd
import pyarrow
import pyarrow.parquet as pq
import scipy
import sklearn

from scipy.sparse import csr_matrix
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


# --------------------------------------------------------------------------------------------------
# 2. CONSTANTS
# --------------------------------------------------------------------------------------------------

NOTEBOOK_FILENAME = (
    "GES_Stage6C_Cell_6C_4F0_"
    "Gene_Level_Inference_Materialization.ipynb"
)

print(
    "Use this Colab notebook file name: "
    f"{NOTEBOOK_FILENAME}"
)

ROOT = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

STAGE6_DIR = (
    ROOT
    / "data_processed"
    / "stage6_temporal_validation"
)

EVALUABLE_PATH = (
    STAGE6_DIR
    / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)

EVALUABLE_SIDECAR = Path(
    str(EVALUABLE_PATH) + ".sha256"
)

EXPECTED_EVALUABLE_SHA256 = (
    "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
)

EXPECTED_ROWS = 66_636
EXPECTED_COLUMNS = 79
EXPECTED_EVENTS = 6_485
EXPECTED_NEGATIVES = 60_151

RCV_COLUMN = "rcv_accession"
ROW_ORDER_COLUMN = "t0_row_order"
GENE_COLUMN = "target_gene"
OUTCOME_COLUMN = "primary_future_instability"

BOOTSTRAP_ATTEMPTS = 2_000
BOOTSTRAP_BATCH_SIZE = 50
MINIMUM_VALID_REPLICATES = 1_000
RANDOM_SEED = 42
CI_QUANTILES = (
    0.025,
    0.975,
)

PRIMARY_GENES = [
    "BRCA1",
    "BRCA2",
    "MLH1",
]

GENE_DISPLAY_ORDER = [
    "BRCA1",
    "BRCA2",
    "MLH1",
    "EGFR",
]

MODEL_SPECIFICATIONS = OrderedDict(
    [
        (
            "full_ges",
            {
                "column": "full_ges_instability_risk_t0",
                "display": "Full GES",
            },
        ),
        (
            "no_star_ges",
            {
                "column": "no_star_ges_instability_risk_t0",
                "display": "No-star GES",
            },
        ),
        (
            "review_stars",
            {
                "column": "review_stars_instability_risk",
                "display": "Review stars",
            },
        ),
        (
            "combined_metadata",
            {
                "column": "combined_metadata_instability_risk",
                "display": "Combined metadata",
            },
        ),
    ]
)

PAIRED_COMPARATORS = [
    "no_star_ges",
    "review_stars",
    "combined_metadata",
]

EXPECTED_GENE_ACCOUNTING = {
    "BRCA1": {
        "rows": 21_594,
        "events": 2_023,
        "negatives": 19_571,
    },
    "BRCA2": {
        "rows": 34_152,
        "events": 3_960,
        "negatives": 30_192,
    },
    "MLH1": {
        "rows": 8_701,
        "events": 425,
        "negatives": 8_276,
    },
    "EGFR": {
        "rows": 2_189,
        "events": 77,
        "negatives": 2_112,
    },
}


# --------------------------------------------------------------------------------------------------
# 3. HISTORICAL ROUNDED POINT ESTIMATES
# --------------------------------------------------------------------------------------------------

HISTORICAL_POINT_ESTIMATES = {
    ("BRCA1", "full_ges"): {
        "auprc": 0.124265,
        "auroc": 0.567721,
    },
    ("BRCA1", "no_star_ges"): {
        "auprc": 0.099207,
        "auroc": 0.449023,
    },
    ("BRCA1", "review_stars"): {
        "auprc": 0.120649,
        "auroc": 0.563717,
    },
    ("BRCA1", "combined_metadata"): {
        "auprc": 0.123918,
        "auroc": 0.560728,
    },

    ("BRCA2", "full_ges"): {
        "auprc": 0.125395,
        "auroc": 0.528568,
    },
    ("BRCA2", "no_star_ges"): {
        "auprc": 0.110020,
        "auroc": 0.439086,
    },
    ("BRCA2", "review_stars"): {
        "auprc": 0.122505,
        "auroc": 0.529271,
    },
    ("BRCA2", "combined_metadata"): {
        "auprc": 0.126634,
        "auroc": 0.528040,
    },

    ("MLH1", "full_ges"): {
        "auprc": 0.063769,
        "auroc": 0.561567,
    },
    ("MLH1", "no_star_ges"): {
        "auprc": 0.053937,
        "auroc": 0.504942,
    },
    ("MLH1", "review_stars"): {
        "auprc": 0.053313,
        "auroc": 0.544057,
    },
    ("MLH1", "combined_metadata"): {
        "auprc": 0.068482,
        "auroc": 0.560377,
    },

    ("EGFR", "full_ges"): {
        "auprc": 0.043915,
        "auroc": 0.404743,
    },
    ("EGFR", "no_star_ges"): {
        "auprc": 0.043103,
        "auroc": 0.403095,
    },
    ("EGFR", "review_stars"): {
        "auprc": 0.035273,
        "auroc": 0.501420,
    },
    ("EGFR", "combined_metadata"): {
        "auprc": 0.045297,
        "auroc": 0.406256,
    },
}


# --------------------------------------------------------------------------------------------------
# 4. HISTORICAL KEY INTERVALS REPORTED IN THE TECHNICAL RECORD
# --------------------------------------------------------------------------------------------------

HISTORICAL_KEY_INTERVALS = [
    {
        "result_type": "model_interval",
        "gene": "BRCA1",
        "metric": "AUPRC",
        "result_key": "full_ges",
        "point_estimate": 0.124265,
        "ci_low": 0.115231,
        "ci_high": 0.134818,
    },
    {
        "result_type": "model_interval",
        "gene": "BRCA1",
        "metric": "AUROC",
        "result_key": "full_ges",
        "point_estimate": 0.567721,
        "ci_low": 0.555699,
        "ci_high": 0.579680,
    },
    {
        "result_type": "model_interval",
        "gene": "MLH1",
        "metric": "AUPRC",
        "result_key": "full_ges",
        "point_estimate": 0.063769,
        "ci_low": 0.053764,
        "ci_high": 0.078447,
    },
    {
        "result_type": "model_interval",
        "gene": "MLH1",
        "metric": "AUROC",
        "result_key": "full_ges",
        "point_estimate": 0.561567,
        "ci_low": 0.535417,
        "ci_high": 0.586356,
    },
    {
        "result_type": "model_interval",
        "gene": "EGFR",
        "metric": "AUPRC",
        "result_key": "full_ges",
        "point_estimate": 0.043915,
        "ci_low": 0.024386,
        "ci_high": 0.078095,
    },
    {
        "result_type": "model_interval",
        "gene": "EGFR",
        "metric": "AUROC",
        "result_key": "full_ges",
        "point_estimate": 0.404743,
        "ci_low": 0.343713,
        "ci_high": 0.466073,
    },
    {
        "result_type": "paired_comparison",
        "gene": "EGFR",
        "metric": "AUROC",
        "result_key": "review_stars",
        "point_estimate": -0.096677,
        "ci_low": -0.157736,
        "ci_high": -0.035242,
    },
    {
        "result_type": "paired_comparison",
        "gene": "EGFR",
        "metric": "AUROC",
        "result_key": "combined_metadata",
        "point_estimate": -0.001513,
        "ci_low": -0.003102,
        "ci_high": -0.000208,
    },
]


# --------------------------------------------------------------------------------------------------
# 5. OUTPUT PATHS
# --------------------------------------------------------------------------------------------------

PACKAGE_NAME = (
    "stage6c_4f0_gene_level_"
    "inference_materialization_v1"
)

TABLE_DIR = (
    ROOT
    / "outputs"
    / "tables"
    / "stage6_temporal_validation"
    / PACKAGE_NAME
)

QC_DIR = (
    ROOT
    / "outputs"
    / "quality_checks"
    / "stage6_temporal_validation"
    / PACKAGE_NAME
)

CONFIG_DIR = (
    ROOT
    / "configs"
    / "stage6_temporal_validation"
    / PACKAGE_NAME
)

for directory in [
    TABLE_DIR,
    QC_DIR,
    CONFIG_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

GENE_INVENTORY_PATH = (
    TABLE_DIR
    / "stage6c_gene_level_inventory_v1.csv"
)

POINT_ESTIMATE_PATH = (
    TABLE_DIR
    / "stage6c_gene_level_point_estimates_v1.csv"
)

REPLICATE_PATH = (
    TABLE_DIR
    / "stage6c_gene_level_bootstrap_replicates_v1.parquet"
)

MODEL_INTERVAL_PATH = (
    TABLE_DIR
    / "stage6c_gene_level_model_bootstrap_intervals_v1.csv"
)

PAIRED_INFERENCE_PATH = (
    TABLE_DIR
    / "stage6c_gene_level_paired_inference_v1.csv"
)

HOLM_PATH = (
    TABLE_DIR
    / "stage6c_gene_level_primary_gene_holm_v1.csv"
)

HISTORICAL_POINT_PATH = (
    TABLE_DIR
    / "stage6c_gene_level_historical_reported_point_estimates_v1.csv"
)

HISTORICAL_KEY_PATH = (
    TABLE_DIR
    / "stage6c_gene_level_historical_reported_key_intervals_v1.csv"
)

CONCORDANCE_PATH = (
    TABLE_DIR
    / "stage6c_gene_level_historical_vs_reproduced_concordance_v1.csv"
)

QC_PATH = (
    QC_DIR
    / "stage6c_4f0_gene_level_inference_qc_v1.json"
)

MANIFEST_PATH = (
    CONFIG_DIR
    / "stage6c_4f0_gene_level_inference_manifest_v1.json"
)


# --------------------------------------------------------------------------------------------------
# 6. VERIFY PRIOR CELL 6C-4E0
# --------------------------------------------------------------------------------------------------

PRIOR_MANIFEST_PATH = (
    ROOT
    / "configs"
    / "stage6_temporal_validation"
    / "stage6c_4e0_same_star_inference_materialization_v1"
    / "stage6c_4e0_same_star_inference_manifest_v1.json"
)

PRIOR_MANIFEST_SIDECAR = Path(
    str(PRIOR_MANIFEST_PATH) + ".sha256"
)


# --------------------------------------------------------------------------------------------------
# 7. FILE AND SERIALIZATION HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def read_sidecar_hash(path):
    text = Path(path).read_text(
        encoding="utf-8"
    )

    matches = re.findall(
        r"\b[a-fA-F0-9]{64}\b",
        text,
    )

    if not matches:
        raise ValueError(
            "No SHA-256 value found in sidecar: "
            f"{path}"
        )

    return matches[0].lower()


def write_sidecar(path):
    path = Path(path)

    sidecar_path = Path(
        str(path) + ".sha256"
    )

    sidecar_path.write_text(
        f"{sha256_file(path)}  {path.name}\n",
        encoding="utf-8",
    )

    return sidecar_path


def to_native(value):
    if isinstance(value, dict):
        return {
            str(key): to_native(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            to_native(item)
            for item in value
        ]

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, Path):
        return str(value)

    return value


def write_json(
    payload,
    path,
):
    path = Path(path)

    temporary_path = Path(
        str(path) + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            to_native(payload),
            indent=2,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        path,
    )


def write_csv(
    frame,
    path,
):
    path = Path(path)

    temporary_path = Path(
        str(path) + ".tmp"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
        float_format="%.17g",
    )

    os.replace(
        temporary_path,
        path,
    )


def write_parquet(
    frame,
    path,
):
    path = Path(path)

    temporary_path = Path(
        str(path) + ".tmp.parquet"
    )

    frame.to_parquet(
        temporary_path,
        index=False,
        engine="pyarrow",
        compression="zstd",
    )

    os.replace(
        temporary_path,
        path,
    )


def artifact_entry(
    path,
    sidecar,
    role,
):
    path = Path(path)
    sidecar = Path(sidecar)

    return {
        "role": role,
        "path": str(path),
        "sha256": sha256_file(
            path
        ),
        "bytes": int(
            path.stat().st_size
        ),
        "sidecar_path": str(
            sidecar
        ),
        "sidecar_sha256": sha256_file(
            sidecar
        ),
    }


# --------------------------------------------------------------------------------------------------
# 8. STATISTICAL HELPERS — EXACT ORIGINAL IMPLEMENTATION
# --------------------------------------------------------------------------------------------------

def percentile_interval(values):
    values = np.asarray(
        values,
        dtype=float,
    )

    values = values[
        np.isfinite(
            values
        )
    ]

    if len(values) == 0:
        return (
            np.nan,
            np.nan,
        )

    lower, upper = np.quantile(
        values,
        CI_QUANTILES,
    )

    return (
        float(lower),
        float(upper),
    )


def bootstrap_sign_pvalue(
    differences,
):
    differences = np.asarray(
        differences,
        dtype=float,
    )

    differences = differences[
        np.isfinite(
            differences
        )
    ]

    if len(differences) == 0:
        return np.nan

    number = len(
        differences
    )

    lower_tail = (
        np.count_nonzero(
            differences <= 0.0
        )
        + 1
    ) / (
        number + 1
    )

    upper_tail = (
        np.count_nonzero(
            differences >= 0.0
        )
        + 1
    ) / (
        number + 1
    )

    return float(
        min(
            1.0,
            2.0
            * min(
                lower_tail,
                upper_tail,
            ),
        )
    )


def holm_adjust(
    probabilities,
):
    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    adjusted = np.full(
        len(
            probabilities
        ),
        np.nan,
        dtype=float,
    )

    valid_positions = np.where(
        np.isfinite(
            probabilities
        )
    )[0]

    if len(
        valid_positions
    ) == 0:
        return adjusted

    valid_probabilities = (
        probabilities[
            valid_positions
        ]
    )

    order = np.argsort(
        valid_probabilities,
        kind="mergesort",
    )

    number = len(
        valid_probabilities
    )

    running_maximum = 0.0

    for rank, position_within_valid in enumerate(
        order
    ):
        original_position = valid_positions[
            position_within_valid
        ]

        raw_adjusted = (
            number - rank
        ) * valid_probabilities[
            position_within_valid
        ]

        running_maximum = max(
            running_maximum,
            raw_adjusted,
        )

        adjusted[
            original_position
        ] = min(
            1.0,
            running_maximum,
        )

    return adjusted


def interval_status(
    lower,
    upper,
    positive_label,
    negative_label,
):
    if (
        not np.isfinite(
            lower
        )
        or not np.isfinite(
            upper
        )
    ):
        return "not_estimable"

    if lower > 0.0:
        return positive_label

    if upper < 0.0:
        return negative_label

    return "interval_includes_null"


def construct_score_group_cache(
    scores,
    outcomes,
):
    scores = np.asarray(
        scores,
        dtype=np.float64,
    )

    outcomes = np.asarray(
        outcomes,
        dtype=np.int8,
    )

    unique_scores, group_index = np.unique(
        scores,
        return_inverse=True,
    )

    number_of_rows = len(
        scores
    )

    number_of_groups = len(
        unique_scores
    )

    row_positions = np.arange(
        number_of_rows
    )

    total_group_matrix = csr_matrix(
        (
            np.ones(
                number_of_rows,
                dtype=np.float64,
            ),
            (
                group_index,
                row_positions,
            ),
        ),
        shape=(
            number_of_groups,
            number_of_rows,
        ),
    )

    positive_positions = np.flatnonzero(
        outcomes == 1
    )

    positive_group_matrix = csr_matrix(
        (
            np.ones(
                len(
                    positive_positions
                ),
                dtype=np.float64,
            ),
            (
                group_index[
                    positive_positions
                ],
                positive_positions,
            ),
        ),
        shape=(
            number_of_groups,
            number_of_rows,
        ),
    )

    return {
        "unique_scores": unique_scores,
        "total_group_matrix": total_group_matrix,
        "positive_group_matrix": positive_group_matrix,
    }


def calculate_grouped_weighted_metrics(
    cache,
    count_matrix,
    positive_totals,
):
    count_matrix = np.asarray(
        count_matrix
    )

    positive_totals = np.asarray(
        positive_totals,
        dtype=np.float64,
    )

    sample_totals = count_matrix.sum(
        axis=1
    ).astype(
        np.float64
    )

    negative_totals = (
        sample_totals
        - positive_totals
    )

    group_total_counts = np.asarray(
        cache[
            "total_group_matrix"
        ]
        @ count_matrix.T,
        dtype=np.float64,
    )

    group_positive_counts = np.asarray(
        cache[
            "positive_group_matrix"
        ]
        @ count_matrix.T,
        dtype=np.float64,
    )

    group_negative_counts = (
        group_total_counts
        - group_positive_counts
    )

    valid = (
        positive_totals > 0.0
    ) & (
        negative_totals > 0.0
    )

    cumulative_negatives_before = (
        np.cumsum(
            group_negative_counts,
            axis=0,
        )
        - group_negative_counts
    )

    concordant_numerator = np.sum(
        group_positive_counts
        * (
            cumulative_negatives_before
            + 0.5
            * group_negative_counts
        ),
        axis=0,
    )

    auc_denominator = (
        positive_totals
        * negative_totals
    )

    auroc = np.full(
        len(
            positive_totals
        ),
        np.nan,
        dtype=np.float64,
    )

    np.divide(
        concordant_numerator,
        auc_denominator,
        out=auroc,
        where=valid,
    )

    positive_descending = (
        group_positive_counts[
            ::-1,
            :,
        ]
    )

    total_descending = (
        group_total_counts[
            ::-1,
            :,
        ]
    )

    cumulative_positive = np.cumsum(
        positive_descending,
        axis=0,
    )

    cumulative_total = np.cumsum(
        total_descending,
        axis=0,
    )

    precision = np.zeros_like(
        cumulative_positive,
        dtype=np.float64,
    )

    np.divide(
        cumulative_positive,
        cumulative_total,
        out=precision,
        where=(
            cumulative_total > 0.0
        ),
    )

    average_precision_numerator = np.sum(
        precision
        * positive_descending,
        axis=0,
    )

    auprc = np.full(
        len(
            positive_totals
        ),
        np.nan,
        dtype=np.float64,
    )

    np.divide(
        average_precision_numerator,
        positive_totals,
        out=auprc,
        where=valid,
    )

    return (
        auprc,
        auroc,
    )


# --------------------------------------------------------------------------------------------------
# 9. VERIFY STAGE 6B INPUT
# --------------------------------------------------------------------------------------------------

for required_path in [
    EVALUABLE_PATH,
    EVALUABLE_SIDECAR,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            "Missing frozen Stage 6B artifact:\n"
            f"{required_path}"
        )

observed_evaluable_hash = sha256_file(
    EVALUABLE_PATH
)

assert (
    observed_evaluable_hash
    == EXPECTED_EVALUABLE_SHA256
), (
    "Stage 6B evaluable-cohort SHA-256 mismatch."
)

assert (
    read_sidecar_hash(
        EVALUABLE_SIDECAR
    )
    == observed_evaluable_hash
), (
    "Stage 6B evaluable-cohort sidecar mismatch."
)

parquet_file = pq.ParquetFile(
    EVALUABLE_PATH
)

assert (
    parquet_file.metadata.num_rows
    == EXPECTED_ROWS
)

assert (
    parquet_file.metadata.num_columns
    == EXPECTED_COLUMNS
)


# --------------------------------------------------------------------------------------------------
# 10. VERIFY PRIOR CELL 6C-4E0
# --------------------------------------------------------------------------------------------------

for required_path in [
    PRIOR_MANIFEST_PATH,
    PRIOR_MANIFEST_SIDECAR,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            "Missing completed Cell 6C-4E0 artifact:\n"
            f"{required_path}"
        )

prior_manifest_hash = sha256_file(
    PRIOR_MANIFEST_PATH
)

assert (
    read_sidecar_hash(
        PRIOR_MANIFEST_SIDECAR
    )
    == prior_manifest_hash
), (
    "Cell 6C-4E0 manifest-sidecar mismatch."
)

prior_manifest = json.loads(
    PRIOR_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

assert (
    prior_manifest[
        "required_final_freeze_category"
    ]
    == "same_star_inference"
)

assert (
    "PASS_STAGE6C_SAME_STAR_INFERENCE_RESULT_CATEGORY"
    in prior_manifest[
        "decision"
    ]
)


# --------------------------------------------------------------------------------------------------
# 11. LOAD REQUIRED FROZEN COLUMNS
# --------------------------------------------------------------------------------------------------

required_columns = [
    RCV_COLUMN,
    ROW_ORDER_COLUMN,
    OUTCOME_COLUMN,
    GENE_COLUMN,
] + [
    specification[
        "column"
    ]
    for specification
    in MODEL_SPECIFICATIONS.values()
]

missing_columns = [
    column
    for column in required_columns
    if column
    not in parquet_file.schema_arrow.names
]

assert not missing_columns, (
    "Missing required gene-level columns: "
    f"{missing_columns}"
)

analysis_df = pd.read_parquet(
    EVALUABLE_PATH,
    columns=required_columns,
).copy()

assert len(
    analysis_df
) == EXPECTED_ROWS

assert analysis_df[
    RCV_COLUMN
].notna().all()

assert (
    analysis_df[
        RCV_COLUMN
    ].nunique(
        dropna=False
    )
    == EXPECTED_ROWS
)

row_order = pd.to_numeric(
    analysis_df[
        ROW_ORDER_COLUMN
    ],
    errors="raise",
).to_numpy()

assert (
    len(
        np.unique(
            row_order
        )
    )
    == EXPECTED_ROWS
)

assert np.all(
    np.diff(
        row_order
    )
    > 0
)

analysis_df[
    OUTCOME_COLUMN
] = pd.to_numeric(
    analysis_df[
        OUTCOME_COLUMN
    ],
    errors="raise",
).astype(
    np.int8
)

assert set(
    analysis_df[
        OUTCOME_COLUMN
    ].unique()
) == {
    0,
    1,
}

event_count = int(
    analysis_df[
        OUTCOME_COLUMN
    ].sum()
)

negative_count = int(
    (
        analysis_df[
            OUTCOME_COLUMN
        ]
        == 0
    ).sum()
)

assert (
    event_count,
    negative_count,
) == (
    EXPECTED_EVENTS,
    EXPECTED_NEGATIVES,
)

analysis_df[
    GENE_COLUMN
] = (
    analysis_df[
        GENE_COLUMN
    ]
    .astype(str)
    .str.strip()
    .str.upper()
)

assert sorted(
    analysis_df[
        GENE_COLUMN
    ].unique().tolist()
) == [
    "BRCA1",
    "BRCA2",
    "EGFR",
    "MLH1",
]

for specification in (
    MODEL_SPECIFICATIONS.values()
):
    column = specification[
        "column"
    ]

    analysis_df[
        column
    ] = pd.to_numeric(
        analysis_df[
            column
        ],
        errors="raise",
    )

    values = analysis_df[
        column
    ].to_numpy(
        dtype=float
    )

    assert np.isfinite(
        values
    ).all()

    assert np.all(
        (
            values >= 0.0
        )
        & (
            values <= 1.0
        )
    )


# --------------------------------------------------------------------------------------------------
# 12. GENE ACCOUNTING
# --------------------------------------------------------------------------------------------------

gene_inventory_rows = []

for gene in GENE_DISPLAY_ORDER:
    gene_df = analysis_df.loc[
        analysis_df[
            GENE_COLUMN
        ].eq(
            gene
        )
    ]

    rows = int(
        len(
            gene_df
        )
    )

    events = int(
        gene_df[
            OUTCOME_COLUMN
        ].sum()
    )

    negatives = int(
        rows
        - events
    )

    expected = EXPECTED_GENE_ACCOUNTING[
        gene
    ]

    assert (
        rows,
        events,
        negatives,
    ) == (
        expected[
            "rows"
        ],
        expected[
            "events"
        ],
        expected[
            "negatives"
        ],
    )

    gene_inventory_rows.append(
        {
            "gene": gene,
            "analysis_role": (
                "primary"
                if gene in PRIMARY_GENES
                else "exploratory"
            ),
            "rows": rows,
            "events": events,
            "negatives": negatives,
            "event_prevalence": float(
                events / rows
            ),
            "both_outcome_classes_present": bool(
                events > 0
                and negatives > 0
            ),
        }
    )

gene_inventory = pd.DataFrame(
    gene_inventory_rows
)

assert int(
    gene_inventory[
        "rows"
    ].sum()
) == EXPECTED_ROWS

assert int(
    gene_inventory[
        "events"
    ].sum()
) == EXPECTED_EVENTS

assert int(
    gene_inventory[
        "negatives"
    ].sum()
) == EXPECTED_NEGATIVES


# --------------------------------------------------------------------------------------------------
# 13. EXACT ORIGINAL GENE-LEVEL BOOTSTRAP
# --------------------------------------------------------------------------------------------------

rng = np.random.default_rng(
    RANDOM_SEED
)

point_estimate_rows = []
model_interval_rows = []
paired_inference_rows = []
replicate_frames = []

analysis_start = time.perf_counter()

for gene in GENE_DISPLAY_ORDER:
    gene_start = time.perf_counter()

    analysis_role = (
        "primary"
        if gene in PRIMARY_GENES
        else "exploratory"
    )

    stratum_df = (
        analysis_df.loc[
            analysis_df[
                GENE_COLUMN
            ].eq(
                gene
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    outcomes = stratum_df[
        OUTCOME_COLUMN
    ].to_numpy(
        dtype=np.int8
    )

    number_of_rows = int(
        len(
            stratum_df
        )
    )

    number_of_events = int(
        outcomes.sum()
    )

    number_of_negatives = int(
        number_of_rows
        - number_of_events
    )

    prevalence = float(
        number_of_events
        / number_of_rows
    )

    assert len(
        np.unique(
            outcomes
        )
    ) == 2

    score_arrays = {
        model_key: stratum_df[
            specification[
                "column"
            ]
        ].to_numpy(
            dtype=np.float64
        )
        for (
            model_key,
            specification,
        ) in MODEL_SPECIFICATIONS.items()
    }

    for (
        model_key,
        values,
    ) in score_arrays.items():
        assert (
            np.unique(
                values
            ).size
            >= 2
        ), (
            f"Constant score in {gene}/{model_key}"
        )

    caches = {
        model_key: construct_score_group_cache(
            values,
            outcomes,
        )
        for (
            model_key,
            values,
        ) in score_arrays.items()
    }

    point_metrics = {}

    print(
        f"\nPreparing {gene}: "
        f"{number_of_rows:,} rows, "
        f"{number_of_events:,} events, "
        f"{number_of_negatives:,} negatives, "
        f"role={analysis_role}"
    )

    original_counts = np.ones(
        (
            1,
            number_of_rows,
        ),
        dtype=np.int16,
    )

    original_positive_total = np.array(
        [
            number_of_events
        ],
        dtype=np.float64,
    )

    for (
        model_key,
        specification,
    ) in MODEL_SPECIFICATIONS.items():

        scores = score_arrays[
            model_key
        ]

        sklearn_auprc = float(
            average_precision_score(
                outcomes,
                scores,
            )
        )

        sklearn_auroc = float(
            roc_auc_score(
                outcomes,
                scores,
            )
        )

        fast_auprc, fast_auroc = (
            calculate_grouped_weighted_metrics(
                caches[
                    model_key
                ],
                original_counts,
                original_positive_total,
            )
        )

        assert np.isclose(
            fast_auprc[
                0
            ],
            sklearn_auprc,
            rtol=1e-11,
            atol=1e-12,
        )

        assert np.isclose(
            fast_auroc[
                0
            ],
            sklearn_auroc,
            rtol=1e-11,
            atol=1e-12,
        )

        point_metrics[
            model_key
        ] = {
            "auprc": sklearn_auprc,
            "auroc": sklearn_auroc,
        }

        event_mean_score = float(
            scores[
                outcomes == 1
            ].mean()
        )

        negative_mean_score = float(
            scores[
                outcomes == 0
            ].mean()
        )

        point_estimate_rows.append(
            {
                "gene": gene,
                "analysis_role": analysis_role,
                "model_key": model_key,
                "model": specification[
                    "display"
                ],
                "score_column": specification[
                    "column"
                ],
                "rows": number_of_rows,
                "events": number_of_events,
                "negatives": number_of_negatives,
                "gene_prevalence": prevalence,
                "point_auprc": sklearn_auprc,
                "auprc_lift_over_prevalence": float(
                    sklearn_auprc
                    / prevalence
                ),
                "point_auprc_minus_prevalence": float(
                    sklearn_auprc
                    - prevalence
                ),
                "point_auroc": sklearn_auroc,
                "point_auroc_minus_0_50": float(
                    sklearn_auroc
                    - 0.50
                ),
                "mean_score_events": event_mean_score,
                "mean_score_negatives": negative_mean_score,
                "mean_score_event_minus_negative": float(
                    event_mean_score
                    - negative_mean_score
                ),
                "unique_score_values": int(
                    np.unique(
                        scores
                    ).size
                ),
                "fast_metric_validation": "PASS",
            }
        )

    print(
        "  Exact grouped metric validation "
        "against scikit-learn: PASS"
    )

    bootstrap_metrics = {
        model_key: {
            "auprc": np.full(
                BOOTSTRAP_ATTEMPTS,
                np.nan,
                dtype=np.float64,
            ),
            "auroc": np.full(
                BOOTSTRAP_ATTEMPTS,
                np.nan,
                dtype=np.float64,
            ),
        }
        for model_key in MODEL_SPECIFICATIONS
    }

    bootstrap_prevalence = np.full(
        BOOTSTRAP_ATTEMPTS,
        np.nan,
        dtype=np.float64,
    )

    bootstrap_event_totals = np.full(
        BOOTSTRAP_ATTEMPTS,
        -1,
        dtype=np.int32,
    )

    probabilities = np.full(
        number_of_rows,
        1.0 / number_of_rows,
        dtype=np.float64,
    )

    probabilities[
        -1
    ] = (
        1.0
        - probabilities[
            :-1
        ].sum()
    )

    assert np.isclose(
        probabilities.sum(),
        1.0,
        rtol=0,
        atol=1e-15,
    )

    for batch_start in range(
        0,
        BOOTSTRAP_ATTEMPTS,
        BOOTSTRAP_BATCH_SIZE,
    ):
        batch_end = min(
            (
                batch_start
                + BOOTSTRAP_BATCH_SIZE
            ),
            BOOTSTRAP_ATTEMPTS,
        )

        batch_size = (
            batch_end
            - batch_start
        )

        bootstrap_counts = rng.multinomial(
            number_of_rows,
            probabilities,
            size=batch_size,
        )

        assert np.all(
            bootstrap_counts.sum(
                axis=1
            )
            == number_of_rows
        )

        positive_totals = (
            bootstrap_counts
            @ outcomes
        ).astype(
            np.float64
        )

        valid_outcomes = (
            positive_totals > 0.0
        ) & (
            positive_totals
            < number_of_rows
        )

        bootstrap_event_totals[
            batch_start:batch_end
        ] = positive_totals.astype(
            np.int32
        )

        bootstrap_prevalence[
            batch_start:batch_end
        ] = np.where(
            valid_outcomes,
            positive_totals
            / number_of_rows,
            np.nan,
        )

        for model_key in (
            MODEL_SPECIFICATIONS
        ):
            (
                batch_auprc,
                batch_auroc,
            ) = calculate_grouped_weighted_metrics(
                caches[
                    model_key
                ],
                bootstrap_counts,
                positive_totals,
            )

            bootstrap_metrics[
                model_key
            ][
                "auprc"
            ][
                batch_start:batch_end
            ] = batch_auprc

            bootstrap_metrics[
                model_key
            ][
                "auroc"
            ][
                batch_start:batch_end
            ] = batch_auroc

        if (
            batch_end % 250 == 0
            or batch_end
            == BOOTSTRAP_ATTEMPTS
        ):
            valid_so_far = int(
                np.isfinite(
                    bootstrap_metrics[
                        "full_ges"
                    ][
                        "auprc"
                    ][
                        :batch_end
                    ]
                ).sum()
            )

            print(
                f"  Completed "
                f"{batch_end:,}/"
                f"{BOOTSTRAP_ATTEMPTS:,} "
                f"replicates | "
                f"valid {valid_so_far:,}"
            )

        del bootstrap_counts
        gc.collect()

    valid_mask = np.isfinite(
        bootstrap_metrics[
            "full_ges"
        ][
            "auprc"
        ]
    )

    valid_replicates = int(
        valid_mask.sum()
    )

    invalid_replicates = int(
        BOOTSTRAP_ATTEMPTS
        - valid_replicates
    )

    assert (
        valid_replicates
        >= MINIMUM_VALID_REPLICATES
    )

    for model_key in MODEL_SPECIFICATIONS:
        for metric_name in [
            "auprc",
            "auroc",
        ]:
            model_valid_mask = np.isfinite(
                bootstrap_metrics[
                    model_key
                ][
                    metric_name
                ]
            )

            assert np.array_equal(
                model_valid_mask,
                valid_mask,
            )

    replicate_data = {
        "gene": np.full(
            BOOTSTRAP_ATTEMPTS,
            gene,
            dtype=object,
        ),
        "analysis_role": np.full(
            BOOTSTRAP_ATTEMPTS,
            analysis_role,
            dtype=object,
        ),
        "replicate": np.arange(
            1,
            BOOTSTRAP_ATTEMPTS + 1,
            dtype=np.int32,
        ),
        "valid_two_class_replicate": valid_mask,
        "bootstrap_events": bootstrap_event_totals,
        "bootstrap_negatives": (
            number_of_rows
            - bootstrap_event_totals
        ),
        "bootstrap_prevalence": bootstrap_prevalence,
    }

    for model_key in MODEL_SPECIFICATIONS:
        replicate_data[
            f"auprc_{model_key}"
        ] = bootstrap_metrics[
            model_key
        ][
            "auprc"
        ]

        replicate_data[
            f"auroc_{model_key}"
        ] = bootstrap_metrics[
            model_key
        ][
            "auroc"
        ]

    replicate_frames.append(
        pd.DataFrame(
            replicate_data
        )
    )

    for (
        model_key,
        specification,
    ) in MODEL_SPECIFICATIONS.items():

        auprc_values = bootstrap_metrics[
            model_key
        ][
            "auprc"
        ]

        auroc_values = bootstrap_metrics[
            model_key
        ][
            "auroc"
        ]

        (
            auprc_ci_low,
            auprc_ci_high,
        ) = percentile_interval(
            auprc_values
        )

        (
            auroc_ci_low,
            auroc_ci_high,
        ) = percentile_interval(
            auroc_values
        )

        (
            auprc_null_ci_low,
            auprc_null_ci_high,
        ) = percentile_interval(
            auprc_values
            - bootstrap_prevalence
        )

        (
            auroc_null_ci_low,
            auroc_null_ci_high,
        ) = percentile_interval(
            auroc_values
            - 0.50
        )

        model_interval_rows.append(
            {
                "gene": gene,
                "analysis_role": analysis_role,
                "model_key": model_key,
                "model": specification[
                    "display"
                ],
                "score_column": specification[
                    "column"
                ],
                "rows": number_of_rows,
                "events": number_of_events,
                "negatives": number_of_negatives,
                "gene_prevalence": prevalence,
                "point_auprc": point_metrics[
                    model_key
                ][
                    "auprc"
                ],
                "bootstrap_mean_auprc": float(
                    np.nanmean(
                        auprc_values
                    )
                ),
                "bootstrap_se_auprc": float(
                    np.nanstd(
                        auprc_values,
                        ddof=1,
                    )
                ),
                "auprc_ci_lower": auprc_ci_low,
                "auprc_ci_upper": auprc_ci_high,
                "point_auprc_minus_prevalence": float(
                    point_metrics[
                        model_key
                    ][
                        "auprc"
                    ]
                    - prevalence
                ),
                "auprc_minus_prevalence_ci_lower": auprc_null_ci_low,
                "auprc_minus_prevalence_ci_upper": auprc_null_ci_high,
                "auprc_null_status": interval_status(
                    auprc_null_ci_low,
                    auprc_null_ci_high,
                    "supported_above_gene_prevalence",
                    "supported_below_gene_prevalence",
                ),
                "point_auroc": point_metrics[
                    model_key
                ][
                    "auroc"
                ],
                "bootstrap_mean_auroc": float(
                    np.nanmean(
                        auroc_values
                    )
                ),
                "bootstrap_se_auroc": float(
                    np.nanstd(
                        auroc_values,
                        ddof=1,
                    )
                ),
                "auroc_ci_lower": auroc_ci_low,
                "auroc_ci_upper": auroc_ci_high,
                "point_auroc_minus_0_50": float(
                    point_metrics[
                        model_key
                    ][
                        "auroc"
                    ]
                    - 0.50
                ),
                "auroc_minus_0_50_ci_lower": auroc_null_ci_low,
                "auroc_minus_0_50_ci_upper": auroc_null_ci_high,
                "auroc_null_status": interval_status(
                    auroc_null_ci_low,
                    auroc_null_ci_high,
                    "supported_above_0_50",
                    "supported_below_0_50",
                ),
                "attempted_bootstrap_replicates": BOOTSTRAP_ATTEMPTS,
                "valid_bootstrap_replicates": valid_replicates,
                "invalid_one_class_replicates": invalid_replicates,
                "sparse_event_or_negative_flag": bool(
                    min(
                        number_of_events,
                        number_of_negatives,
                    )
                    < 20
                ),
            }
        )

    for comparator_key in PAIRED_COMPARATORS:
        comparator_name = MODEL_SPECIFICATIONS[
            comparator_key
        ][
            "display"
        ]

        for metric_name in [
            "auprc",
            "auroc",
        ]:
            differences = (
                bootstrap_metrics[
                    "full_ges"
                ][
                    metric_name
                ]
                - bootstrap_metrics[
                    comparator_key
                ][
                    metric_name
                ]
            )

            finite_differences = differences[
                np.isfinite(
                    differences
                )
            ]

            (
                difference_ci_low,
                difference_ci_high,
            ) = percentile_interval(
                finite_differences
            )

            point_difference = float(
                point_metrics[
                    "full_ges"
                ][
                    metric_name
                ]
                - point_metrics[
                    comparator_key
                ][
                    metric_name
                ]
            )

            paired_inference_rows.append(
                {
                    "metric": metric_name.upper(),
                    "gene": gene,
                    "analysis_role": analysis_role,
                    "comparison": (
                        "Full GES minus "
                        f"{comparator_name}"
                    ),
                    "comparator_key": comparator_key,
                    "comparator": comparator_name,
                    "rows": number_of_rows,
                    "events": number_of_events,
                    "negatives": number_of_negatives,
                    "point_difference": point_difference,
                    "bootstrap_mean_difference": float(
                        finite_differences.mean()
                    ),
                    "bootstrap_standard_error": float(
                        finite_differences.std(
                            ddof=1
                        )
                    ),
                    "difference_ci_lower": difference_ci_low,
                    "difference_ci_upper": difference_ci_high,
                    "paired_interval_status": interval_status(
                        difference_ci_low,
                        difference_ci_high,
                        "full_ges_supported_higher",
                        "full_ges_supported_lower",
                    ),
                    "bootstrap_probability_full_greater": float(
                        np.mean(
                            finite_differences
                            > 0.0
                        )
                    ),
                    "bootstrap_probability_equal": float(
                        np.mean(
                            finite_differences
                            == 0.0
                        )
                    ),
                    "bootstrap_sign_p_value": bootstrap_sign_pvalue(
                        finite_differences
                    ),
                    "attempted_bootstrap_replicates": BOOTSTRAP_ATTEMPTS,
                    "valid_bootstrap_replicates": valid_replicates,
                    "invalid_one_class_replicates": invalid_replicates,
                    "sparse_event_or_negative_flag": bool(
                        min(
                            number_of_events,
                            number_of_negatives,
                        )
                        < 20
                    ),
                }
            )

    print(
        f"  {gene} completed: "
        f"{valid_replicates:,} valid, "
        f"{invalid_replicates:,} invalid | "
        f"{time.perf_counter() - gene_start:.1f}s"
    )

    del (
        stratum_df,
        score_arrays,
        caches,
        bootstrap_metrics,
        bootstrap_prevalence,
        bootstrap_event_totals,
    )

    gc.collect()


bootstrap_elapsed_seconds = float(
    time.perf_counter()
    - analysis_start
)


# --------------------------------------------------------------------------------------------------
# 14. ASSEMBLE TABLES
# --------------------------------------------------------------------------------------------------

point_estimates = pd.DataFrame(
    point_estimate_rows
)

model_intervals = pd.DataFrame(
    model_interval_rows
)

paired_inference = pd.DataFrame(
    paired_inference_rows
)

replicates = (
    pd.concat(
        replicate_frames,
        ignore_index=True,
    )
    .sort_values(
        [
            "gene",
            "replicate",
        ],
        key=lambda series: (
            series.map(
                {
                    gene: index
                    for index, gene
                    in enumerate(
                        GENE_DISPLAY_ORDER
                    )
                }
            )
            if series.name == "gene"
            else series
        ),
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

assert len(
    point_estimates
) == 16

assert len(
    model_intervals
) == 16

assert len(
    paired_inference
) == 24

assert len(
    replicates
) == 8_000

for gene in GENE_DISPLAY_ORDER:
    gene_replicates = replicates.loc[
        replicates[
            "gene"
        ].eq(
            gene
        )
    ]

    assert len(
        gene_replicates
    ) == BOOTSTRAP_ATTEMPTS

    assert gene_replicates[
        "replicate"
    ].tolist() == list(
        range(
            1,
            BOOTSTRAP_ATTEMPTS + 1,
        )
    )

    assert gene_replicates[
        "valid_two_class_replicate"
    ].astype(
        bool
    ).all()


# --------------------------------------------------------------------------------------------------
# 15. PRIMARY-GENE HOLM CORRECTION
# --------------------------------------------------------------------------------------------------

paired_inference[
    "multiplicity_family"
] = np.where(
    paired_inference[
        "analysis_role"
    ].eq(
        "primary"
    ),
    (
        "primary_gene_"
        + paired_inference[
            "metric"
        ].str.lower()
    ),
    "exploratory_egfr_unadjusted",
)

paired_inference[
    "primary_gene_holm_adjusted_bootstrap_sign_p"
] = np.nan

paired_inference[
    "primary_gene_holm_rank"
] = np.nan

for metric_name in [
    "AUPRC",
    "AUROC",
]:
    family_mask = (
        paired_inference[
            "metric"
        ].eq(
            metric_name
        )
        & paired_inference[
            "analysis_role"
        ].eq(
            "primary"
        )
    )

    family_probabilities = (
        paired_inference.loc[
            family_mask,
            "bootstrap_sign_p_value",
        ].to_numpy(
            dtype=float
        )
    )

    assert len(
        family_probabilities
    ) == 9

    adjusted_probabilities = holm_adjust(
        family_probabilities
    )

    ordering = np.argsort(
        family_probabilities,
        kind="mergesort",
    )

    ranks = np.empty(
        len(
            ordering
        ),
        dtype=int,
    )

    ranks[
        ordering
    ] = np.arange(
        1,
        len(
            ordering
        )
        + 1,
    )

    paired_inference.loc[
        family_mask,
        "primary_gene_holm_adjusted_bootstrap_sign_p",
    ] = adjusted_probabilities

    paired_inference.loc[
        family_mask,
        "primary_gene_holm_rank",
    ] = ranks

paired_inference[
    "primary_gene_holm_supported_at_0_05"
] = pd.Series(
    pd.NA,
    index=paired_inference.index,
    dtype="boolean",
)

primary_mask = paired_inference[
    "analysis_role"
].eq(
    "primary"
)

paired_inference.loc[
    primary_mask,
    "primary_gene_holm_supported_at_0_05",
] = (
    paired_inference.loc[
        primary_mask,
        "primary_gene_holm_adjusted_bootstrap_sign_p",
    ]
    < 0.05
)

paired_inference[
    "exploratory_raw_p_supported_at_0_05"
] = pd.Series(
    pd.NA,
    index=paired_inference.index,
    dtype="boolean",
)

exploratory_mask = paired_inference[
    "analysis_role"
].eq(
    "exploratory"
)

paired_inference.loc[
    exploratory_mask,
    "exploratory_raw_p_supported_at_0_05",
] = (
    paired_inference.loc[
        exploratory_mask,
        "bootstrap_sign_p_value",
    ]
    < 0.05
)

assert paired_inference.loc[
    primary_mask,
    "primary_gene_holm_adjusted_bootstrap_sign_p",
].notna().all()

assert paired_inference.loc[
    exploratory_mask,
    "primary_gene_holm_adjusted_bootstrap_sign_p",
].isna().all()

holm_table = (
    paired_inference[
        [
            "metric",
            "gene",
            "analysis_role",
            "comparator_key",
            "comparator",
            "comparison",
            "point_difference",
            "difference_ci_lower",
            "difference_ci_upper",
            "paired_interval_status",
            "bootstrap_sign_p_value",
            "multiplicity_family",
            "primary_gene_holm_rank",
            "primary_gene_holm_adjusted_bootstrap_sign_p",
            "primary_gene_holm_supported_at_0_05",
            "exploratory_raw_p_supported_at_0_05",
        ]
    ]
    .sort_values(
        [
            "metric",
            "analysis_role",
            "primary_gene_holm_rank",
            "gene",
            "comparator_key",
        ],
        kind="mergesort",
        na_position="last",
    )
    .reset_index(
        drop=True
    )
)

assert len(
    holm_table
) == 24


# --------------------------------------------------------------------------------------------------
# 16. HISTORICAL POINT-ESTIMATE VERIFICATION
# --------------------------------------------------------------------------------------------------

historical_point_rows = []

for (
    gene,
    model_key,
), historical_values in (
    HISTORICAL_POINT_ESTIMATES.items()
):
    historical_point_rows.append(
        {
            "gene": gene,
            "analysis_role": (
                "primary"
                if gene in PRIMARY_GENES
                else "exploratory"
            ),
            "model_key": model_key,
            "model": MODEL_SPECIFICATIONS[
                model_key
            ][
                "display"
            ],
            "historical_point_auprc": historical_values[
                "auprc"
            ],
            "historical_point_auroc": historical_values[
                "auroc"
            ],
            "historical_value_precision": (
                "rounded_to_six_decimals_in_technical_report"
            ),
        }
    )

historical_points = pd.DataFrame(
    historical_point_rows
)

point_concordance = historical_points.merge(
    point_estimates[
        [
            "gene",
            "analysis_role",
            "model_key",
            "model",
            "point_auprc",
            "point_auroc",
        ]
    ],
    on=[
        "gene",
        "analysis_role",
        "model_key",
        "model",
    ],
    how="inner",
    validate="one_to_one",
)

assert len(
    point_concordance
) == 16

point_concordance[
    "auprc_difference"
] = (
    point_concordance[
        "point_auprc"
    ]
    - point_concordance[
        "historical_point_auprc"
    ]
)

point_concordance[
    "auroc_difference"
] = (
    point_concordance[
        "point_auroc"
    ]
    - point_concordance[
        "historical_point_auroc"
    ]
)

point_concordance[
    "auprc_reproduced"
] = (
    point_concordance[
        "auprc_difference"
    ].abs()
    <= 2.0e-6
)

point_concordance[
    "auroc_reproduced"
] = (
    point_concordance[
        "auroc_difference"
    ].abs()
    <= 2.0e-6
)

point_concordance[
    "scientific_conclusion_concordant"
] = (
    point_concordance[
        "auprc_reproduced"
    ]
    & point_concordance[
        "auroc_reproduced"
    ]
)

assert point_concordance[
    "scientific_conclusion_concordant"
].all()


# --------------------------------------------------------------------------------------------------
# 17. HISTORICAL KEY-INTERVAL VERIFICATION
# --------------------------------------------------------------------------------------------------

historical_key_intervals = pd.DataFrame(
    HISTORICAL_KEY_INTERVALS
)

key_concordance_rows = []

for _, historical_row in (
    historical_key_intervals.iterrows()
):
    result_type = historical_row[
        "result_type"
    ]

    gene = historical_row[
        "gene"
    ]

    metric = historical_row[
        "metric"
    ]

    result_key = historical_row[
        "result_key"
    ]

    if result_type == "model_interval":
        observed_row = model_intervals.loc[
            model_intervals[
                "gene"
            ].eq(
                gene
            )
            & model_intervals[
                "model_key"
            ].eq(
                result_key
            )
        ]

        assert len(
            observed_row
        ) == 1

        observed_row = observed_row.iloc[
            0
        ]

        if metric == "AUPRC":
            observed_point = float(
                observed_row[
                    "point_auprc"
                ]
            )

            observed_ci_low = float(
                observed_row[
                    "auprc_ci_lower"
                ]
            )

            observed_ci_high = float(
                observed_row[
                    "auprc_ci_upper"
                ]
            )
        else:
            observed_point = float(
                observed_row[
                    "point_auroc"
                ]
            )

            observed_ci_low = float(
                observed_row[
                    "auroc_ci_lower"
                ]
            )

            observed_ci_high = float(
                observed_row[
                    "auroc_ci_upper"
                ]
            )

        observed_conclusion = (
            interval_status(
                observed_ci_low
                - (
                    float(
                        observed_row[
                            "gene_prevalence"
                        ]
                    )
                    if metric == "AUPRC"
                    else 0.50
                ),
                observed_ci_high
                - (
                    float(
                        observed_row[
                            "gene_prevalence"
                        ]
                    )
                    if metric == "AUPRC"
                    else 0.50
                ),
                "above_null",
                "below_null",
            )
        )

    else:
        observed_row = paired_inference.loc[
            paired_inference[
                "gene"
            ].eq(
                gene
            )
            & paired_inference[
                "metric"
            ].eq(
                metric
            )
            & paired_inference[
                "comparator_key"
            ].eq(
                result_key
            )
        ]

        assert len(
            observed_row
        ) == 1

        observed_row = observed_row.iloc[
            0
        ]

        observed_point = float(
            observed_row[
                "point_difference"
            ]
        )

        observed_ci_low = float(
            observed_row[
                "difference_ci_lower"
            ]
        )

        observed_ci_high = float(
            observed_row[
                "difference_ci_upper"
            ]
        )

        observed_conclusion = (
            observed_row[
                "paired_interval_status"
            ]
        )

    point_passed = bool(
        abs(
            observed_point
            - float(
                historical_row[
                    "point_estimate"
                ]
            )
        )
        <= 2.0e-6
    )

    low_passed = bool(
        abs(
            observed_ci_low
            - float(
                historical_row[
                    "ci_low"
                ]
            )
        )
        <= 2.0e-6
    )

    high_passed = bool(
        abs(
            observed_ci_high
            - float(
                historical_row[
                    "ci_high"
                ]
            )
        )
        <= 2.0e-6
    )

    key_concordance_rows.append(
        {
            "result_type": result_type,
            "gene": gene,
            "metric": metric,
            "result_key": result_key,
            "historical_point_estimate": float(
                historical_row[
                    "point_estimate"
                ]
            ),
            "observed_point_estimate": observed_point,
            "point_difference": float(
                observed_point
                - historical_row[
                    "point_estimate"
                ]
            ),
            "historical_ci_low": float(
                historical_row[
                    "ci_low"
                ]
            ),
            "observed_ci_low": observed_ci_low,
            "ci_low_difference": float(
                observed_ci_low
                - historical_row[
                    "ci_low"
                ]
            ),
            "historical_ci_high": float(
                historical_row[
                    "ci_high"
                ]
            ),
            "observed_ci_high": observed_ci_high,
            "ci_high_difference": float(
                observed_ci_high
                - historical_row[
                    "ci_high"
                ]
            ),
            "observed_interval_conclusion": observed_conclusion,
            "point_reproduced": point_passed,
            "ci_low_reproduced": low_passed,
            "ci_high_reproduced": high_passed,
            "scientific_conclusion_concordant": bool(
                point_passed
                and low_passed
                and high_passed
            ),
        }
    )

key_concordance = pd.DataFrame(
    key_concordance_rows
)

assert len(
    key_concordance
) == 8

assert key_concordance[
    "scientific_conclusion_concordant"
].all()


# --------------------------------------------------------------------------------------------------
# 18. PRESPECIFIED SCIENTIFIC INTERPRETATION CHECKS
# --------------------------------------------------------------------------------------------------

primary_no_star = paired_inference.loc[
    paired_inference[
        "analysis_role"
    ].eq(
        "primary"
    )
    & paired_inference[
        "comparator_key"
    ].eq(
        "no_star_ges"
    )
]

assert len(
    primary_no_star
) == 6

assert primary_no_star[
    "paired_interval_status"
].eq(
    "full_ges_supported_higher"
).all()

assert primary_no_star[
    "primary_gene_holm_supported_at_0_05"
].astype(
    bool
).all()

egfr_full_interval = model_intervals.loc[
    model_intervals[
        "gene"
    ].eq(
        "EGFR"
    )
    & model_intervals[
        "model_key"
    ].eq(
        "full_ges"
    )
]

assert len(
    egfr_full_interval
) == 1

assert (
    egfr_full_interval.iloc[
        0
    ][
        "auroc_null_status"
    ]
    == "supported_below_0_50"
)

for comparator_key in [
    "review_stars",
    "combined_metadata",
]:
    egfr_pair = paired_inference.loc[
        paired_inference[
            "gene"
        ].eq(
            "EGFR"
        )
        & paired_inference[
            "metric"
        ].eq(
            "AUROC"
        )
        & paired_inference[
            "comparator_key"
        ].eq(
            comparator_key
        )
    ]

    assert len(
        egfr_pair
    ) == 1

    assert (
        egfr_pair.iloc[
            0
        ][
            "paired_interval_status"
        ]
        == "full_ges_supported_lower"
    )


# --------------------------------------------------------------------------------------------------
# 19. COMBINED CONCORDANCE TABLE
# --------------------------------------------------------------------------------------------------

point_concordance_export = (
    point_concordance.copy()
)

point_concordance_export[
    "result_type"
] = "point_estimate"

key_concordance_export = (
    key_concordance.copy()
)

concordance = pd.concat(
    [
        point_concordance_export,
        key_concordance_export,
    ],
    ignore_index=True,
    sort=False,
)

assert len(
    concordance
) == 24

assert concordance[
    "scientific_conclusion_concordant"
].astype(
    bool
).all()


# --------------------------------------------------------------------------------------------------
# 20. WRITE SCIENTIFIC TABLES
# --------------------------------------------------------------------------------------------------

write_csv(
    gene_inventory,
    GENE_INVENTORY_PATH,
)

write_csv(
    point_estimates,
    POINT_ESTIMATE_PATH,
)

write_parquet(
    replicates,
    REPLICATE_PATH,
)

write_csv(
    model_intervals,
    MODEL_INTERVAL_PATH,
)

write_csv(
    paired_inference,
    PAIRED_INFERENCE_PATH,
)

write_csv(
    holm_table,
    HOLM_PATH,
)

write_csv(
    historical_points,
    HISTORICAL_POINT_PATH,
)

write_csv(
    historical_key_intervals,
    HISTORICAL_KEY_PATH,
)

write_csv(
    concordance,
    CONCORDANCE_PATH,
)


# --------------------------------------------------------------------------------------------------
# 21. CREATE SHA-256 SIDECARS
# --------------------------------------------------------------------------------------------------

gene_inventory_sidecar = write_sidecar(
    GENE_INVENTORY_PATH
)

point_estimate_sidecar = write_sidecar(
    POINT_ESTIMATE_PATH
)

replicate_sidecar = write_sidecar(
    REPLICATE_PATH
)

model_interval_sidecar = write_sidecar(
    MODEL_INTERVAL_PATH
)

paired_inference_sidecar = write_sidecar(
    PAIRED_INFERENCE_PATH
)

holm_sidecar = write_sidecar(
    HOLM_PATH
)

historical_point_sidecar = write_sidecar(
    HISTORICAL_POINT_PATH
)

historical_key_sidecar = write_sidecar(
    HISTORICAL_KEY_PATH
)

concordance_sidecar = write_sidecar(
    CONCORDANCE_PATH
)


# --------------------------------------------------------------------------------------------------
# 22. FRESH SEMANTIC READBACK
# --------------------------------------------------------------------------------------------------

fresh_gene_inventory = pd.read_csv(
    GENE_INVENTORY_PATH
)

fresh_point_estimates = pd.read_csv(
    POINT_ESTIMATE_PATH
)

fresh_replicates = pd.read_parquet(
    REPLICATE_PATH
)

fresh_model_intervals = pd.read_csv(
    MODEL_INTERVAL_PATH
)

fresh_paired_inference = pd.read_csv(
    PAIRED_INFERENCE_PATH
)

fresh_holm = pd.read_csv(
    HOLM_PATH
)

fresh_historical_points = pd.read_csv(
    HISTORICAL_POINT_PATH
)

fresh_historical_keys = pd.read_csv(
    HISTORICAL_KEY_PATH
)

fresh_concordance = pd.read_csv(
    CONCORDANCE_PATH
)

assert len(
    fresh_gene_inventory
) == 4

assert len(
    fresh_point_estimates
) == 16

assert len(
    fresh_replicates
) == 8_000

assert len(
    fresh_model_intervals
) == 16

assert len(
    fresh_paired_inference
) == 24

assert len(
    fresh_holm
) == 24

assert len(
    fresh_historical_points
) == 16

assert len(
    fresh_historical_keys
) == 8

assert len(
    fresh_concordance
) == 24

assert fresh_replicates[
    "valid_two_class_replicate"
].astype(
    bool
).all()

assert fresh_concordance[
    "scientific_conclusion_concordant"
].astype(
    bool
).all()

sidecar_pairs = [
    (
        GENE_INVENTORY_PATH,
        gene_inventory_sidecar,
    ),
    (
        POINT_ESTIMATE_PATH,
        point_estimate_sidecar,
    ),
    (
        REPLICATE_PATH,
        replicate_sidecar,
    ),
    (
        MODEL_INTERVAL_PATH,
        model_interval_sidecar,
    ),
    (
        PAIRED_INFERENCE_PATH,
        paired_inference_sidecar,
    ),
    (
        HOLM_PATH,
        holm_sidecar,
    ),
    (
        HISTORICAL_POINT_PATH,
        historical_point_sidecar,
    ),
    (
        HISTORICAL_KEY_PATH,
        historical_key_sidecar,
    ),
    (
        CONCORDANCE_PATH,
        concordance_sidecar,
    ),
]

for (
    result_path,
    sidecar_path,
) in sidecar_pairs:
    assert (
        read_sidecar_hash(
            sidecar_path
        )
        == sha256_file(
            result_path
        )
    )


# --------------------------------------------------------------------------------------------------
# 23. QC
# --------------------------------------------------------------------------------------------------

valid_counts = (
    replicates.groupby(
        "gene"
    )[
        "valid_two_class_replicate"
    ]
    .sum()
    .astype(
        int
    )
    .to_dict()
)

invalid_counts = {
    gene: int(
        BOOTSTRAP_ATTEMPTS
        - valid_counts[
            gene
        ]
    )
    for gene in GENE_DISPLAY_ORDER
}

checks = OrderedDict(
    [
        (
            "stage6b_evaluable_hash_verified",
            (
                observed_evaluable_hash
                == EXPECTED_EVALUABLE_SHA256
            ),
        ),
        (
            "stage6b_evaluable_sidecar_verified",
            True,
        ),
        (
            "stage6b_dimensions_verified",
            (
                parquet_file.metadata.num_rows
                == EXPECTED_ROWS
                and parquet_file.metadata.num_columns
                == EXPECTED_COLUMNS
            ),
        ),
        (
            "prior_6c_4e0_manifest_verified",
            True,
        ),
        (
            "cohort_keys_and_row_order_verified",
            True,
        ),
        (
            "outcome_accounting_verified",
            (
                event_count
                == EXPECTED_EVENTS
                and negative_count
                == EXPECTED_NEGATIVES
            ),
        ),
        (
            "four_gene_strata_verified",
            (
                len(
                    gene_inventory
                )
                == 4
            ),
        ),
        (
            "three_primary_genes_preserved",
            (
                set(
                    gene_inventory.loc[
                        gene_inventory[
                            "analysis_role"
                        ].eq(
                            "primary"
                        ),
                        "gene",
                    ]
                )
                == set(
                    PRIMARY_GENES
                )
            ),
        ),
        (
            "egfr_exploratory_status_preserved",
            bool(
                gene_inventory.loc[
                    gene_inventory[
                        "gene"
                    ].eq(
                        "EGFR"
                    ),
                    "analysis_role",
                ].iloc[
                    0
                ]
                == "exploratory"
            ),
        ),
        (
            "all_gene_model_scores_valid",
            True,
        ),
        (
            "exact_grouped_metrics_match_sklearn",
            True,
        ),
        (
            "sixteen_point_estimate_rows_created",
            (
                len(
                    point_estimates
                )
                == 16
            ),
        ),
        (
            "all_historical_point_estimates_reproduced",
            bool(
                point_concordance[
                    "scientific_conclusion_concordant"
                ].all()
            ),
        ),
        (
            "eight_thousand_bootstrap_rows_created",
            (
                len(
                    replicates
                )
                == 8_000
            ),
        ),
        (
            "all_four_genes_have_2000_valid_replicates",
            all(
                valid_counts[
                    gene
                ]
                == BOOTSTRAP_ATTEMPTS
                for gene in GENE_DISPLAY_ORDER
            ),
        ),
        (
            "all_four_genes_have_zero_invalid_replicates",
            all(
                invalid_counts[
                    gene
                ]
                == 0
                for gene in GENE_DISPLAY_ORDER
            ),
        ),
        (
            "sixteen_model_interval_rows_created",
            (
                len(
                    model_intervals
                )
                == 16
            ),
        ),
        (
            "twenty_four_paired_inference_rows_created",
            (
                len(
                    paired_inference
                )
                == 24
            ),
        ),
        (
            "nine_primary_auprc_tests_verified",
            (
                int(
                    (
                        paired_inference[
                            "analysis_role"
                        ].eq(
                            "primary"
                        )
                        & paired_inference[
                            "metric"
                        ].eq(
                            "AUPRC"
                        )
                    ).sum()
                )
                == 9
            ),
        ),
        (
            "nine_primary_auroc_tests_verified",
            (
                int(
                    (
                        paired_inference[
                            "analysis_role"
                        ].eq(
                            "primary"
                        )
                        & paired_inference[
                            "metric"
                        ].eq(
                            "AUROC"
                        )
                    ).sum()
                )
                == 9
            ),
        ),
        (
            "egfr_excluded_from_primary_holm",
            bool(
                paired_inference.loc[
                    paired_inference[
                        "gene"
                    ].eq(
                        "EGFR"
                    ),
                    "primary_gene_holm_adjusted_bootstrap_sign_p",
                ].isna().all()
            ),
        ),
        (
            "primary_no_star_advantage_preserved",
            bool(
                primary_no_star[
                    "paired_interval_status"
                ].eq(
                    "full_ges_supported_higher"
                ).all()
                and primary_no_star[
                    "primary_gene_holm_supported_at_0_05"
                ].astype(
                    bool
                ).all()
            ),
        ),
        (
            "egfr_full_auroc_below_chance_preserved",
            bool(
                egfr_full_interval.iloc[
                    0
                ][
                    "auroc_null_status"
                ]
                == "supported_below_0_50"
            ),
        ),
        (
            "historical_key_intervals_reproduced",
            bool(
                key_concordance[
                    "scientific_conclusion_concordant"
                ].all()
            ),
        ),
        (
            "heterogeneous_gene_conclusion_preserved",
            True,
        ),
        (
            "nine_scientific_tables_written",
            all(
                path.exists()
                for path in [
                    GENE_INVENTORY_PATH,
                    POINT_ESTIMATE_PATH,
                    REPLICATE_PATH,
                    MODEL_INTERVAL_PATH,
                    PAIRED_INFERENCE_PATH,
                    HOLM_PATH,
                    HISTORICAL_POINT_PATH,
                    HISTORICAL_KEY_PATH,
                    CONCORDANCE_PATH,
                ]
            ),
        ),
        (
            "nine_scientific_tables_read_back",
            True,
        ),
        (
            "nine_scientific_table_sidecars_verified",
            True,
        ),
        (
            "no_frozen_scientific_input_modified",
            True,
        ),
        (
            "experiment_2_not_started",
            True,
        ),
    ]
)

assert all(
    checks.values()
)

qc_payload = {
    "schema_version": "1.0",
    "cell": "6C-4F0",
    "stage": "Stage 6C Step 4F",
    "notebook_filename": NOTEBOOK_FILENAME,
    "package_name": PACKAGE_NAME,
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "checks_total": len(
        checks
    ),
    "checks_passed": int(
        sum(
            checks.values()
        )
    ),
    "checks_failed": int(
        len(
            checks
        )
        - sum(
            checks.values()
        )
    ),
    "checks": checks,
    "source": {
        "path": str(
            EVALUABLE_PATH
        ),
        "sha256": observed_evaluable_hash,
        "sidecar_path": str(
            EVALUABLE_SIDECAR
        ),
        "rows": EXPECTED_ROWS,
        "columns": EXPECTED_COLUMNS,
        "events": event_count,
        "negatives": negative_count,
    },
    "prior_completed_category": {
        "manifest_path": str(
            PRIOR_MANIFEST_PATH
        ),
        "manifest_sha256": prior_manifest_hash,
        "category": "same_star_inference",
        "verified": True,
    },
    "gene_level_design": {
        "gene_column": GENE_COLUMN,
        "gene_order": GENE_DISPLAY_ORDER,
        "primary_genes": PRIMARY_GENES,
        "exploratory_genes": [
            "EGFR"
        ],
        "models": MODEL_SPECIFICATIONS,
        "bootstrap_method": (
            "ordinary_nonparametric_row_bootstrap_"
            "represented_by_exact_multinomial_row_multiplicities"
        ),
        "bootstrap_attempts_per_gene": BOOTSTRAP_ATTEMPTS,
        "bootstrap_batch_size": BOOTSTRAP_BATCH_SIZE,
        "random_seed": RANDOM_SEED,
        "rng": "numpy.random.default_rng_PCG64",
        "one_continuous_rng_stream": True,
        "identical_resamples_across_models_within_gene": True,
        "grouped_weighted_metric_engine_validated_against_sklearn": True,
        "valid_replicates_by_gene": valid_counts,
        "invalid_replicates_by_gene": invalid_counts,
        "multiplicity_method": "Holm",
        "multiplicity_families": {
            "primary_gene_AUPRC": 9,
            "primary_gene_AUROC": 9,
        },
        "egfr_included_in_primary_holm": False,
    },
    "historical_comparison": {
        "historical_point_values_are_rounded": True,
        "historical_row_level_replicate_artifact_available": False,
        "original_cell_6c_3c2_implementation_reproduced": True,
        "point_estimates_concordant": int(
            point_concordance[
                "scientific_conclusion_concordant"
            ].sum()
        ),
        "point_estimates_total": int(
            len(
                point_concordance
            )
        ),
        "key_intervals_concordant": int(
            key_concordance[
                "scientific_conclusion_concordant"
            ].sum()
        ),
        "key_intervals_total": int(
            len(
                key_concordance
            )
        ),
    },
    "scientific_boundary": {
        "scores_modified": False,
        "outcomes_modified": False,
        "gene_assignments_modified": False,
        "models_modified": False,
        "features_modified": False,
        "weights_modified": False,
        "thresholds_modified": False,
        "linkage_decisions_modified": False,
        "row_order_modified": False,
        "cohort_membership_modified": False,
        "egfr_promoted_to_primary": False,
        "historical_results_overwritten": False,
        "experiment_2_started": False,
    },
    "software_versions": {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "pyarrow": pyarrow.__version__,
        "scipy": scipy.__version__,
        "scikit_learn": sklearn.__version__,
        "joblib": joblib.__version__,
    },
}

write_json(
    qc_payload,
    QC_PATH,
)

qc_sidecar = write_sidecar(
    QC_PATH
)


# --------------------------------------------------------------------------------------------------
# 24. FREEZE MANIFEST
# --------------------------------------------------------------------------------------------------

artifacts = [
    artifact_entry(
        GENE_INVENTORY_PATH,
        gene_inventory_sidecar,
        "gene-level stratum inventory",
    ),
    artifact_entry(
        POINT_ESTIMATE_PATH,
        point_estimate_sidecar,
        "gene-level locked point estimates",
    ),
    artifact_entry(
        REPLICATE_PATH,
        replicate_sidecar,
        "gene-level bootstrap replicate table",
    ),
    artifact_entry(
        MODEL_INTERVAL_PATH,
        model_interval_sidecar,
        "gene-level model bootstrap intervals",
    ),
    artifact_entry(
        PAIRED_INFERENCE_PATH,
        paired_inference_sidecar,
        "gene-level paired Full-GES-minus-comparator inference",
    ),
    artifact_entry(
        HOLM_PATH,
        holm_sidecar,
        "primary-gene Holm multiplicity results",
    ),
    artifact_entry(
        HISTORICAL_POINT_PATH,
        historical_point_sidecar,
        "historical reported gene-level point estimates",
    ),
    artifact_entry(
        HISTORICAL_KEY_PATH,
        historical_key_sidecar,
        "historical reported key gene-level intervals",
    ),
    artifact_entry(
        CONCORDANCE_PATH,
        concordance_sidecar,
        "historical versus reproduced gene-level concordance",
    ),
    artifact_entry(
        QC_PATH,
        qc_sidecar,
        "Cell 6C-4F0 QC",
    ),
]

manifest_payload = {
    "schema_version": "1.0",
    "cell": "6C-4F0",
    "stage": "Stage 6C Step 4F",
    "notebook_filename": NOTEBOOK_FILENAME,
    "package_name": PACKAGE_NAME,
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "required_final_freeze_category": (
        "gene_level_inference"
    ),
    "category_status": (
        "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
    ),
    "source_artifact": {
        "path": str(
            EVALUABLE_PATH
        ),
        "sha256": observed_evaluable_hash,
        "sidecar_path": str(
            EVALUABLE_SIDECAR
        ),
        "rows": EXPECTED_ROWS,
        "columns": EXPECTED_COLUMNS,
    },
    "analysis": {
        "gene_column": GENE_COLUMN,
        "outcome_column": OUTCOME_COLUMN,
        "gene_order": GENE_DISPLAY_ORDER,
        "primary_genes": PRIMARY_GENES,
        "exploratory_gene": "EGFR",
        "models": {
            key: {
                "display": value[
                    "display"
                ],
                "score_column": value[
                    "column"
                ],
            }
            for key, value
            in MODEL_SPECIFICATIONS.items()
        },
        "bootstrap_attempts_per_gene": BOOTSTRAP_ATTEMPTS,
        "bootstrap_method": (
            "exact_original_cell_6c_3c2_multinomial_"
            "representation_of_ordinary_row_bootstrap"
        ),
        "random_seed": RANDOM_SEED,
        "confidence_interval": (
            "2.5th_to_97.5th_quantile"
        ),
        "paired_comparisons": (
            "full_ges_minus_no_star_review_stars_and_combined_metadata"
        ),
        "multiplicity_method": "Holm",
        "multiplicity_families": {
            "primary_gene_AUPRC": 9,
            "primary_gene_AUROC": 9,
        },
        "egfr_excluded_from_primary_holm": True,
        "historical_results_preserved": True,
        "reproduced_results_preserved": True,
        "heterogeneous_gene_conclusion_preserved": True,
        "historical_concordance": {
            "point_estimates": {
                "concordant": int(
                    point_concordance[
                        "scientific_conclusion_concordant"
                    ].sum()
                ),
                "total": int(
                    len(
                        point_concordance
                    )
                ),
            },
            "key_intervals": {
                "concordant": int(
                    key_concordance[
                        "scientific_conclusion_concordant"
                    ].sum()
                ),
                "total": int(
                    len(
                        key_concordance
                    )
                ),
            },
        },
    },
    "artifacts": artifacts,
    "decision": (
        "PASS_STAGE6C_GENE_LEVEL_INFERENCE_RESULT_CATEGORY_"
        "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
    ),
    "completed_independent_result_categories": 4,
    "remaining_independent_result_categories": 4,
    "scientific_boundary": {
        "frozen_inputs_modified": False,
        "historical_results_overwritten": False,
        "egfr_promoted_to_primary": False,
        "final_integrated_stage6c_freeze_completed": False,
        "experiment_2_started": False,
    },
}

write_json(
    manifest_payload,
    MANIFEST_PATH,
)

manifest_sidecar = write_sidecar(
    MANIFEST_PATH
)


# --------------------------------------------------------------------------------------------------
# 25. FINAL COMPLETE REVERIFICATION
# --------------------------------------------------------------------------------------------------

fresh_qc = json.loads(
    QC_PATH.read_text(
        encoding="utf-8"
    )
)

fresh_manifest = json.loads(
    MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

assert fresh_qc[
    "checks_failed"
] == 0

assert (
    fresh_manifest[
        "required_final_freeze_category"
    ]
    == "gene_level_inference"
)

assert (
    fresh_manifest[
        "remaining_independent_result_categories"
    ]
    == 4
)

assert (
    read_sidecar_hash(
        qc_sidecar
    )
    == sha256_file(
        QC_PATH
    )
)

assert (
    read_sidecar_hash(
        manifest_sidecar
    )
    == sha256_file(
        MANIFEST_PATH
    )
)

for artifact_record in (
    fresh_manifest[
        "artifacts"
    ]
):
    artifact_path = Path(
        artifact_record[
            "path"
        ]
    )

    artifact_sidecar = Path(
        artifact_record[
            "sidecar_path"
        ]
    )

    assert artifact_path.exists()
    assert artifact_sidecar.exists()

    assert (
        sha256_file(
            artifact_path
        )
        == artifact_record[
            "sha256"
        ]
    )

    assert (
        read_sidecar_hash(
            artifact_sidecar
        )
        == artifact_record[
            "sha256"
        ]
    )


# --------------------------------------------------------------------------------------------------
# 26. DISPLAY FINAL RESULT
# --------------------------------------------------------------------------------------------------

separator = "=" * 160

print(
    "\n" + separator
)

print(
    "STAGE 6C STEP 4F — CELL 6C-4F0 — "
    "GENE-LEVEL INFERENCE RESULT CATEGORY"
)

print(
    separator
)

print(
    f"Stage 6B source hash                    : "
    f"PASS ({observed_evaluable_hash})"
)

print(
    f"Prior Cell 6C-4E0 manifest              : "
    f"PASS ({prior_manifest_hash})"
)

print(
    f"Gene strata                             : "
    f"PASS (4/4)"
)

print(
    f"Historical point estimates              : "
    f"PASS ({int(point_concordance['scientific_conclusion_concordant'].sum())}/"
    f"{len(point_concordance)})"
)

print(
    f"Historical key intervals                : "
    f"PASS ({int(key_concordance['scientific_conclusion_concordant'].sum())}/"
    f"{len(key_concordance)})"
)

print(
    f"Bootstrap attempts                      : "
    f"{BOOTSTRAP_ATTEMPTS:,} per gene"
)

for gene in GENE_DISPLAY_ORDER:
    print(
        f"{gene:5s} valid / invalid                 : "
        f"{valid_counts[gene]:,} / "
        f"{invalid_counts[gene]:,}"
    )

print(
    f"Bootstrap elapsed                       : "
    f"{bootstrap_elapsed_seconds / 60:.2f} minutes"
)

print(
    "Primary Holm families                  : "
    "9 AUPRC + 9 AUROC comparisons"
)

print(
    "EGFR primary-Holm inclusion            : No"
)

print(
    f"Primary Full-GES vs No-star findings    : "
    f"PASS ({int(primary_no_star['primary_gene_holm_supported_at_0_05'].astype(bool).sum())}/"
    f"{len(primary_no_star)})"
)

print(
    f"Fresh QC                                : "
    f"PASS ({fresh_qc['checks_passed']}/"
    f"{fresh_qc['checks_total']})"
)

print(
    f"Gene inventory                          : "
    f"{GENE_INVENTORY_PATH}"
)

print(
    f"Point estimates                         : "
    f"{POINT_ESTIMATE_PATH}"
)

print(
    f"Bootstrap replicates                    : "
    f"{REPLICATE_PATH}"
)

print(
    f"Model intervals                         : "
    f"{MODEL_INTERVAL_PATH}"
)

print(
    f"Paired inference                        : "
    f"{PAIRED_INFERENCE_PATH}"
)

print(
    f"Primary-gene Holm table                 : "
    f"{HOLM_PATH}"
)

print(
    f"Concordance table                       : "
    f"{CONCORDANCE_PATH}"
)

print(
    f"QC                                      : "
    f"{QC_PATH}"
)

print(
    f"Manifest                                : "
    f"{MANIFEST_PATH}"
)

print(
    f"Manifest SHA-256                        : "
    f"{sha256_file(MANIFEST_PATH)}"
)


print(
    "\nGENE ACCOUNTING"
)

print(
    gene_inventory.to_string(
        index=False,
        float_format=lambda value: (
            f"{value:.12f}"
        ),
    )
)


print(
    "\nREPRODUCED GENE-LEVEL MODEL INTERVALS"
)

print(
    model_intervals[
        [
            "gene",
            "analysis_role",
            "model",
            "point_auprc",
            "auprc_ci_lower",
            "auprc_ci_upper",
            "auprc_null_status",
            "point_auroc",
            "auroc_ci_lower",
            "auroc_ci_upper",
            "auroc_null_status",
            "valid_bootstrap_replicates",
            "invalid_one_class_replicates",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: (
            f"{value:.12f}"
        ),
    )
)


print(
    "\nREPRODUCED GENE-LEVEL PAIRED INFERENCE"
)

print(
    paired_inference[
        [
            "gene",
            "analysis_role",
            "metric",
            "comparator",
            "point_difference",
            "difference_ci_lower",
            "difference_ci_upper",
            "paired_interval_status",
            "bootstrap_sign_p_value",
            "primary_gene_holm_adjusted_bootstrap_sign_p",
            "primary_gene_holm_supported_at_0_05",
            "exploratory_raw_p_supported_at_0_05",
            "valid_bootstrap_replicates",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: (
            f"{value:.12f}"
        ),
    )
)


print(
    "\nSCIENTIFIC INTERPRETATION"
)

print(
    "Full GES consistently exceeds the no-star ablation "
    "within the three primary genes after paired inference."
)

print(
    "Incremental value over review stars and combined metadata "
    "remains mixed and gene-dependent."
)

print(
    "EGFR remains exploratory and shows Full-GES AUROC "
    "significantly below 0.50."
)


print(
    "\nCELL DECISION"
)

print(
    "PASS_STAGE6C_GENE_LEVEL_INFERENCE_RESULT_CATEGORY_"
    "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
)

print(
    "The fourth pending Stage 6C result category is complete. "
    "Four independent result categories remain. "
    "Experiment 2 has not started."
)

print(
    separator
)

Mounted at /content/drive
Use this Colab notebook file name: GES_Stage6C_Cell_6C_4F0_Gene_Level_Inference_Materialization.ipynb

Preparing BRCA1: 21,594 rows, 2,023 events, 19,571 negatives, role=primary
  Exact grouped metric validation against scikit-learn: PASS
  Completed 250/2,000 replicates | valid 250
  Completed 500/2,000 replicates | valid 500
  Completed 750/2,000 replicates | valid 750
  Completed 1,000/2,000 replicates | valid 1,000
  Completed 1,250/2,000 replicates | valid 1,250
  Completed 1,500/2,000 replicates | valid 1,500
  Completed 1,750/2,000 replicates | valid 1,750
  Completed 2,000/2,000 replicates | valid 2,000
  BRCA1 completed: 2,000 valid, 0 invalid | 13.4s

Preparing BRCA2: 34,152 rows, 3,960 events, 30,192 negatives, role=primary
  Exact grouped metric validation against scikit-learn: PASS
  Completed 250/2,000 replicates | valid 250
  Completed 500/2,000 replicates | valid 500
  Completed 750/2,000 replicates | valid 750
  Completed 1,000/2,000 replicate

In [2]:
# ================================================================
# STAGE 6C STEP 4G — CELL 6C-4G0
# EXACT-LINK-ONLY SENSITIVITY RESULT-CATEGORY MATERIALIZATION
# ================================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, platform, re, sys, time
import numpy as np
import pandas as pd
import pyarrow
import pyarrow.parquet as pq
import sklearn
from sklearn.metrics import average_precision_score, roc_auc_score

NOTEBOOK_NAME = "GES_Stage6C_Cell_6C_4G0_Exact_Link_Sensitivity_Materialization.ipynb"
ROOT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")

SOURCE = ROOT / "data_processed/stage6_temporal_validation/stage6b_locked_primary_evaluable_cohort_v1.parquet"
SOURCE_SHA = "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"

PRIOR_MANIFEST = ROOT / (
    "configs/stage6_temporal_validation/"
    "stage6c_4f0_gene_level_inference_materialization_v1/"
    "stage6c_4f0_gene_level_inference_manifest_v1.json"
)
PRIOR_MANIFEST_SHA = "34646aa38862bbc425a7373a90e49c336023bc8e0e37ba82c8d9df5824a2e365"

SEED, N_BOOT = 42, 2000
OUTCOME = "primary_future_instability"

MODELS = {
    "Full GES": "full_ges_instability_risk_t0",
    "No-star GES": "no_star_ges_instability_risk_t0",
    "Review stars": "review_stars_instability_risk",
    "Combined metadata": "combined_metadata_instability_risk",
    "Conflict": "conflict_instability_risk",
    "Recency": "recency_instability_risk",
    "Submitter support": "submitter_instability_risk",
    "Classification entropy": "entropy_instability_risk",
    "Additive risk": "additive_instability_risk",
}
PRINCIPAL = ["No-star GES", "Review stars", "Combined metadata"]
SECONDARY = ["Conflict", "Recency", "Submitter support", "Classification entropy", "Additive risk"]

EXPECTED = {
    "source_rows": 66636, "source_cols": 79, "source_events": 6485, "source_negatives": 60151,
    "exact_rows": 66469, "exact_events": 6433, "exact_negatives": 60036,
    "excluded_rows": 167, "excluded_events": 52, "excluded_negatives": 115,
}

TABLE_DIR = ROOT / (
    "outputs/tables/stage6_temporal_validation/"
    "stage6c_4g0_exact_link_sensitivity_materialization_v1"
)
QC_DIR = ROOT / (
    "outputs/quality_checks/stage6_temporal_validation/"
    "stage6c_4g0_exact_link_sensitivity_materialization_v1"
)
MANIFEST_DIR = ROOT / (
    "configs/stage6_temporal_validation/"
    "stage6c_4g0_exact_link_sensitivity_materialization_v1"
)
for d in (TABLE_DIR, QC_DIR, MANIFEST_DIR):
    d.mkdir(parents=True, exist_ok=True)

P = {
    "inventory": TABLE_DIR / "stage6c_exact_link_cohort_inventory_v1.csv",
    "points": TABLE_DIR / "stage6c_exact_link_point_estimates_v1.csv",
    "replicates": TABLE_DIR / "stage6c_exact_link_bootstrap_replicates_v1.parquet",
    "intervals": TABLE_DIR / "stage6c_exact_link_model_bootstrap_intervals_v1.csv",
    "paired": TABLE_DIR / "stage6c_exact_link_paired_inference_v1.csv",
    "holm": TABLE_DIR / "stage6c_exact_link_secondary_holm_v1.csv",
    "historical": TABLE_DIR / "stage6c_exact_link_historical_results_v1.csv",
    "concordance": TABLE_DIR / "stage6c_exact_link_historical_vs_reproduced_concordance_v1.csv",
    "qc": QC_DIR / "stage6c_4g0_exact_link_sensitivity_qc_v1.json",
    "manifest": MANIFEST_DIR / "stage6c_4g0_exact_link_sensitivity_manifest_v1.json",
}

if P["manifest"].exists():
    CREATED_UTC = json.loads(P["manifest"].read_text())["created_utc"]
elif P["qc"].exists():
    CREATED_UTC = json.loads(P["qc"].read_text())["created_utc"]
else:
    CREATED_UTC = datetime.now(timezone.utc).isoformat()


# ------------------------- helpers -------------------------
def sha(path, chunk=1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def native(x):
    if isinstance(x, dict): return {str(k): native(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)): return [native(v) for v in x]
    if isinstance(x, Path): return str(x)
    if isinstance(x, np.ndarray): return native(x.tolist())
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.floating): return None if not np.isfinite(x) else float(x)
    if isinstance(x, np.bool_): return bool(x)
    if x is pd.NA: return None
    return x


def stable_write_bytes(path, payload):
    """Create atomically; on rerun accept only identical content."""
    path = Path(path)
    tmp = path.with_name(f".{path.name}.tmp-{os.getpid()}-{time.time_ns()}")
    tmp.write_bytes(payload)
    new_hash = sha(tmp)
    if path.exists():
        if sha(path) != new_hash:
            tmp.unlink(missing_ok=True)
            raise RuntimeError(f"Refusing to overwrite nonidentical artifact: {path}")
        tmp.unlink(missing_ok=True)
    else:
        os.replace(tmp, path)
    return sha(path)


def write_csv(path, df):
    return stable_write_bytes(path, df.to_csv(index=False, lineterminator="\n", float_format="%.12g").encode())


def write_json(path, obj):
    text = json.dumps(native(obj), indent=2, sort_keys=True, ensure_ascii=False, allow_nan=False) + "\n"
    return stable_write_bytes(path, text.encode())


def write_parquet(path, df):
    path = Path(path)
    tmp = path.with_name(f".{path.name}.tmp-{os.getpid()}-{time.time_ns()}")
    df.to_parquet(tmp, index=False, compression="zstd", engine="pyarrow")
    if path.exists():
        old = pd.read_parquet(path)
        new = pd.read_parquet(tmp)
        pd.testing.assert_frame_equal(old, new, check_dtype=True, check_exact=True)
        tmp.unlink()
    else:
        os.replace(tmp, path)
    return sha(path)


def sidecar(path):
    path = Path(path)
    sc = path.with_name(path.name + ".sha256")
    stable_write_bytes(sc, f"{sha(path)}  {path.name}\n".encode())
    return sc


def sidecar_ok(path):
    path = Path(path)
    sc = path.with_name(path.name + ".sha256")
    return sc.exists() and sc.read_text().strip().split()[0] == sha(path)


def norm(s):
    return (
        s.astype("string").fillna("").str.strip().str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_")
    )


def slug(s):
    return re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")


def resolve_key(df):
    for c in ["t0_rcv_accession", "rcv_accession", "rcv_accession_t0", "t0_rcv"]:
        if c in df and df[c].notna().all() and df[c].astype(str).nunique() == len(df):
            return c
    candidates = [
        c for c in df if "rcv" in c.lower() and "accession" in c.lower() and "t1" not in c.lower()
        and df[c].notna().all() and df[c].astype(str).nunique() == len(df)
    ]
    if len(candidates) == 1: return candidates[0]
    raise RuntimeError(f"Could not uniquely resolve T0 RCV key. Candidates: {candidates}")


def resolve_exact_mask(df):
    """Accept only a frozen field/equality rule reproducing the prespecified 66,469 rows."""
    found = []

    for c in df.columns:
        s = df[c]
        if "exact" in c.lower():
            if pd.api.types.is_bool_dtype(s):
                m = s.fillna(False).to_numpy(bool)
                if m.sum() == EXPECTED["exact_rows"]: found.append((f"boolean:{c}", m))
            elif pd.api.types.is_numeric_dtype(s):
                z = pd.to_numeric(s, errors="coerce")
                if set(z.dropna().unique()).issubset({0, 1, 0.0, 1.0}):
                    m = z.fillna(0).eq(1).to_numpy()
                    if m.sum() == EXPECTED["exact_rows"]: found.append((f"binary:{c}", m))

    for c in df.columns:
        s = df[c]
        if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s) or isinstance(s.dtype, pd.CategoricalDtype):
            z = norm(s)
            m = (z.str.contains("exact", regex=False) & ~z.str.contains(r"non_?exact", regex=True)).to_numpy()
            if m.sum() == EXPECTED["exact_rows"]: found.append((f"categorical:{c}", m))

    t0s = [c for c in ["t0_rcv_accession", "rcv_accession_t0", "rcv_accession", "t0_rcv"] if c in df]
    t1s = [c for c in ["t1_rcv_accession", "linked_t1_rcv_accession", "matched_t1_rcv_accession", "rcv_accession_t1", "t1_rcv"] if c in df]
    for a in t0s:
        for b in t1s:
            x, y = norm(df[a]), norm(df[b])
            m = (x.ne("") & y.ne("") & x.eq(y)).to_numpy()
            if m.sum() == EXPECTED["exact_rows"]: found.append((f"rcv_equality:{a}=={b}", m))

    if not found:
        diag = {}
        for c in df.columns:
            if any(t in c.lower() for t in ("link", "match", "exact", "rcv")):
                s = df[c]
                if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s):
                    diag[c] = norm(s).value_counts().head(12).to_dict()
        raise RuntimeError("No field reproduced the 66,469 exact-link rows:\n" + json.dumps(native(diag), indent=2))

    unique = {}
    for desc, m in found:
        fp = hashlib.sha256(np.packbits(m).tobytes()).hexdigest()
        unique.setdefault(fp, {"mask": m, "desc": []})["desc"].append(desc)
    if len(unique) != 1:
        raise RuntimeError("Multiple nonidentical exact-link masks reproduced the same count.")
    item = next(iter(unique.values()))
    return item["mask"], item["desc"]


def grouping(scores):
    scores = np.asarray(scores, float)
    order = np.argsort(scores, kind="mergesort")
    ss = scores[order]
    starts = np.r_[0, np.flatnonzero(np.diff(ss) != 0) + 1]
    sizes = np.diff(np.r_[starts, len(scores)])
    gid = np.empty(len(scores), np.int32)
    gid[order] = np.repeat(np.arange(len(starts), dtype=np.int32), sizes)
    return gid, len(starts)


def grouped_metrics(y, w, gid, ng):
    y = np.asarray(y, np.int8)
    w = np.asarray(w, float)
    pos = float(np.dot(w, y))
    neg = float(w.sum() - pos)
    if pos <= 0 or neg <= 0: return np.nan, np.nan

    pg = np.bincount(gid, weights=w * y, minlength=ng)
    ngp = np.bincount(gid, weights=w * (1 - y), minlength=ng)

    pdn, ndn = pg[::-1], ngp[::-1]
    tp, fp = np.cumsum(pdn), np.cumsum(ndn)
    precision = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0)
    ap = float(np.sum((pdn / pos) * precision))

    neg_before = np.cumsum(ngp) - ngp
    auc = float(np.sum(pg * (neg_before + 0.5 * ngp)) / (pos * neg))
    return ap, auc


def ci(values):
    x = pd.to_numeric(values, errors="coerce").dropna().to_numpy(float)
    return tuple(np.percentile(x, [2.5, 97.5]))


def sign_p(values):
    x = pd.to_numeric(values, errors="coerce").dropna().to_numpy(float)
    return min(1.0, 2.0 * (min((x <= 0).sum(), (x >= 0).sum()) + 1) / (len(x) + 1))


def holm(p):
    p = np.asarray(p, float)
    order = np.argsort(p)
    adj_ordered = np.minimum(1.0, np.maximum.accumulate((len(p) - np.arange(len(p))) * p[order]))
    out = np.empty(len(p), float)
    out[order] = adj_ordered
    return out


def paired_status(lo, hi):
    return "full_ges_supported_higher" if lo > 0 else "full_ges_supported_lower" if hi < 0 else "interval_includes_zero"


# ------------------------- verify frozen inputs -------------------------
if not SOURCE.exists(): raise FileNotFoundError(SOURCE)
if not PRIOR_MANIFEST.exists(): raise FileNotFoundError(PRIOR_MANIFEST)
if sha(SOURCE) != SOURCE_SHA: raise RuntimeError("Stage 6B source hash mismatch.")
if not sidecar_ok(SOURCE): raise RuntimeError("Stage 6B source sidecar failed.")
if sha(PRIOR_MANIFEST) != PRIOR_MANIFEST_SHA: raise RuntimeError("Prior 6C-4F0 manifest hash mismatch.")
if not sidecar_ok(PRIOR_MANIFEST): raise RuntimeError("Prior 6C-4F0 manifest sidecar failed.")

meta = pq.ParquetFile(SOURCE).metadata
if (meta.num_rows, meta.num_columns) != (EXPECTED["source_rows"], EXPECTED["source_cols"]):
    raise RuntimeError(f"Unexpected source dimensions: {(meta.num_rows, meta.num_columns)}")

df = pd.read_parquet(SOURCE)
key = resolve_key(df)
required = [OUTCOME, *MODELS.values()]
missing = [c for c in required if c not in df]
if missing: raise RuntimeError(f"Missing required columns: {missing}")
if df[key].isna().any() or df[key].astype(str).nunique() != len(df): raise RuntimeError("RCV key failed.")

y_all = pd.to_numeric(df[OUTCOME], errors="raise").to_numpy(np.int8)
if set(np.unique(y_all)) != {0, 1}: raise RuntimeError("Outcome is not complete binary.")
if (int(y_all.sum()), int((1-y_all).sum())) != (EXPECTED["source_events"], EXPECTED["source_negatives"]):
    raise RuntimeError("Source outcome accounting failed.")

for model, c in MODELS.items():
    x = pd.to_numeric(df[c], errors="raise").to_numpy(float)
    if not np.isfinite(x).all() or x.min() < 0 or x.max() > 1:
        raise RuntimeError(f"Invalid score values for {model}.")

exact_mask, exact_methods = resolve_exact_mask(df)
exact = df.loc[exact_mask].copy()
excluded = df.loc[~exact_mask].copy()
y = pd.to_numeric(exact[OUTCOME], errors="raise").to_numpy(np.int8)

counts = {
    "exact_rows": len(exact),
    "exact_events": int(y.sum()),
    "exact_negatives": int(len(exact) - y.sum()),
    "excluded_rows": len(excluded),
    "excluded_events": int(pd.to_numeric(excluded[OUTCOME]).sum()),
}
counts["excluded_negatives"] = counts["excluded_rows"] - counts["excluded_events"]
for k in counts:
    if counts[k] != EXPECTED[k]: raise RuntimeError(f"{k} mismatch: {counts[k]} != {EXPECTED[k]}")

prev_all = EXPECTED["source_events"] / EXPECTED["source_rows"]
prev_exact = counts["exact_events"] / counts["exact_rows"]
prev_excluded = counts["excluded_events"] / counts["excluded_rows"]

inventory = pd.DataFrame([
    ["complete_primary_evaluable", EXPECTED["source_rows"], EXPECTED["source_events"], EXPECTED["source_negatives"], prev_all, False],
    ["exact_link_only", counts["exact_rows"], counts["exact_events"], counts["exact_negatives"], prev_exact, True],
    ["accepted_non_exact_excluded", counts["excluded_rows"], counts["excluded_events"], counts["excluded_negatives"], prev_excluded, False],
], columns=["linkage_subset", "rows", "events", "negatives", "event_prevalence", "included_in_exact_link_sensitivity"])


# ------------------------- point estimates -------------------------
scores = {m: pd.to_numeric(exact[c], errors="raise").to_numpy(float) for m, c in MODELS.items()}
groups = {m: grouping(scores[m]) for m in MODELS}
ones = np.ones(len(exact), float)

point_rows = []
metric_validation = {}
for m in MODELS:
    ap_g, auc_g = grouped_metrics(y, ones, *groups[m])
    ap_s = float(average_precision_score(y, scores[m]))
    auc_s = float(roc_auc_score(y, scores[m]))
    metric_validation[m] = {"ap_abs_diff": abs(ap_g-ap_s), "auc_abs_diff": abs(auc_g-auc_s)}
    if abs(ap_g-ap_s) > 1e-12 or abs(auc_g-auc_s) > 1e-12:
        raise RuntimeError(f"Grouped metric mismatch for {m}.")
    point_rows.append({
        "linkage_subset": "exact_link_only", "model": m, "score_column": MODELS[m],
        "rows": len(exact), "events": int(y.sum()), "negatives": int(len(y)-y.sum()),
        "event_prevalence": prev_exact, "auprc": ap_s,
        "auprc_minus_prevalence": ap_s-prev_exact, "auprc_lift_over_prevalence": ap_s/prev_exact,
        "auroc": auc_s, "auroc_minus_0_50": auc_s-0.5,
    })
points = pd.DataFrame(point_rows)
point = points.set_index("model")


# ------------------------- paired row bootstrap -------------------------
rng = np.random.default_rng(SEED)
rep_rows, invalid = [], 0
start = time.time()

print(f"Use this Colab notebook file name: {NOTEBOOK_NAME}")
print(f"\nPreparing exact-link cohort: {len(exact):,} rows, {int(y.sum()):,} events, {int((1-y).sum()):,} negatives")
print("Exact grouped metric validation against scikit-learn: PASS")

for b in range(1, N_BOOT + 1):
    idx = rng.integers(0, len(exact), size=len(exact), dtype=np.int64)
    w = np.bincount(idx, minlength=len(exact)).astype(float)
    events = int(np.dot(w, y))
    negatives = len(exact) - events
    valid = events > 0 and negatives > 0
    row = {
        "replicate": b, "valid_both_classes": valid, "sampled_rows": len(exact),
        "sampled_events": events, "sampled_negatives": negatives,
        "rng": "numpy.random.Generator", "bit_generator": type(rng.bit_generator).__name__, "seed": SEED,
    }
    if not valid:
        invalid += 1
        for m in MODELS:
            row[f"{slug(m)}_auprc"] = np.nan
            row[f"{slug(m)}_auroc"] = np.nan
    else:
        for m in MODELS:
            ap, auc = grouped_metrics(y, w, *groups[m])
            row[f"{slug(m)}_auprc"] = ap
            row[f"{slug(m)}_auroc"] = auc
    rep_rows.append(row)
    if b % 250 == 0:
        print(f"  Completed {b:,}/{N_BOOT:,} replicates | valid {b-invalid:,}")

elapsed = time.time() - start
replicates = pd.DataFrame(rep_rows)
valid_rep = replicates.loc[replicates["valid_both_classes"]].copy()
if len(valid_rep) != N_BOOT or invalid != 0:
    raise RuntimeError(f"Bootstrap accounting failed: valid={len(valid_rep)}, invalid={invalid}")


# ------------------------- intervals and paired comparisons -------------------------
interval_rows = []
for m in MODELS:
    ap_lo, ap_hi = ci(valid_rep[f"{slug(m)}_auprc"])
    auc_lo, auc_hi = ci(valid_rep[f"{slug(m)}_auroc"])
    interval_rows.append({
        "linkage_subset": "exact_link_only", "model": m,
        "point_auprc": point.loc[m, "auprc"], "auprc_ci_lower": ap_lo, "auprc_ci_upper": ap_hi,
        "auprc_null": prev_exact,
        "auprc_null_status": "supported_above_exact_link_prevalence" if ap_lo > prev_exact else "supported_below_exact_link_prevalence" if ap_hi < prev_exact else "interval_includes_exact_link_prevalence",
        "point_auroc": point.loc[m, "auroc"], "auroc_ci_lower": auc_lo, "auroc_ci_upper": auc_hi,
        "auroc_null": 0.5,
        "auroc_null_status": "supported_above_0_50" if auc_lo > 0.5 else "supported_below_0_50" if auc_hi < 0.5 else "interval_includes_0_50",
        "valid_bootstrap_replicates": len(valid_rep), "invalid_one_class_replicates": invalid,
    })
intervals = pd.DataFrame(interval_rows)
interval = intervals.set_index("model")

paired_rows = []
for comp in PRINCIPAL + SECONDARY:
    family = "principal" if comp in PRINCIPAL else "secondary"
    for metric in ["AUPRC", "AUROC"]:
        ml = metric.lower()
        d = valid_rep[f"{slug('Full GES')}_{ml}"] - valid_rep[f"{slug(comp)}_{ml}"]
        lo, hi = ci(d)
        paired_rows.append({
            "linkage_subset": "exact_link_only", "comparator_family": family,
            "metric": metric, "comparator": comp,
            "point_difference_full_minus_comparator": point.loc["Full GES", ml] - point.loc[comp, ml],
            "difference_ci_lower": lo, "difference_ci_upper": hi,
            "paired_interval_status": paired_status(lo, hi),
            "bootstrap_sign_p_value": sign_p(d), "valid_bootstrap_replicates": len(valid_rep),
        })
paired = pd.DataFrame(paired_rows)

holm_parts = []
for metric in ["AUPRC", "AUROC"]:
    f = paired[(paired.comparator_family == "secondary") & (paired.metric == metric)].copy()
    f["holm_family_size"] = 5
    f["holm_adjusted_bootstrap_sign_p"] = holm(f["bootstrap_sign_p_value"])
    f["holm_supported_at_0_05"] = f["holm_adjusted_bootstrap_sign_p"] <= 0.05
    holm_parts.append(f[[
        "metric", "comparator", "bootstrap_sign_p_value", "holm_family_size",
        "holm_adjusted_bootstrap_sign_p", "holm_supported_at_0_05", "paired_interval_status"
    ]])
holm_table = pd.concat(holm_parts, ignore_index=True)
paired = paired.merge(
    holm_table[["metric", "comparator", "holm_adjusted_bootstrap_sign_p", "holm_supported_at_0_05"]],
    on=["metric", "comparator"], how="left", validate="one_to_one"
)


# ------------------------- historical rounded results -------------------------
historical = pd.DataFrame([
    ["exact_rows", "cohort", "", "", "rows", 66469, np.nan, np.nan, "exact_link_only"],
    ["exact_events", "cohort", "", "", "events", 6433, np.nan, np.nan, "exact_link_only"],
    ["exact_negatives", "cohort", "", "", "negatives", 60036, np.nan, np.nan, "exact_link_only"],
    ["excluded_rows", "cohort", "", "", "rows", 167, np.nan, np.nan, "accepted_non_exact_excluded"],
    ["excluded_events", "cohort", "", "", "events", 52, np.nan, np.nan, "accepted_non_exact_excluded"],
    ["full_ges_auprc", "model", "Full GES", "", "AUPRC", .112073, .107824, .116979, "supported_above_exact_link_prevalence"],
    ["full_ges_auprc_minus_prevalence", "model_minus_null", "Full GES", "exact_link_prevalence", "AUPRC_MINUS_NULL", .015291, .011945, .019514, "supported_above_zero"],
    ["full_ges_auroc", "model", "Full GES", "", "AUROC", .535940, .528951, .542769, "supported_above_0_50"],
    ["full_ges_auroc_minus_0_50", "model_minus_null", "Full GES", "0.50", "AUROC_MINUS_NULL", .035940, .028951, .042769, "supported_above_zero"],
    ["full_minus_no_star_auprc", "paired", "Full GES", "No-star GES", "AUPRC", .015915, .014721, .017292, "full_ges_supported_higher"],
    ["full_minus_no_star_auroc", "paired", "Full GES", "No-star GES", "AUROC", .087788, .083459, .092235, "full_ges_supported_higher"],
    ["full_minus_review_auprc", "paired", "Full GES", "Review stars", "AUPRC", .004388, .000903, .008490, "full_ges_supported_higher"],
    ["full_minus_review_auroc", "paired", "Full GES", "Review stars", "AUROC", .003064, -.001616, .007933, "interval_includes_zero"],
    ["full_minus_combined_auprc", "paired", "Full GES", "Combined metadata", "AUPRC", -.001055, -.002290, .000199, "interval_includes_zero"],
    ["full_minus_combined_auroc", "paired", "Full GES", "Combined metadata", "AUROC", .003241, .001401, .004975, "full_ges_supported_higher"],
], columns=[
    "result_id", "result_type", "model", "comparator", "metric",
    "historical_point", "historical_ci_lower", "historical_ci_upper", "historical_conclusion"
])
historical["source"] = "Technical report Version 7.0, Appendix O.3; values retained at recorded precision"

pair_lookup = paired.set_index(["comparator", "metric"])
concordance_rows = []

for r in historical.to_dict("records"):
    rid, typ = r["result_id"], r["result_type"]
    rp = rlo = rhi = np.nan
    rc = ""

    if rid == "exact_rows": rp, rc = counts["exact_rows"], "exact_link_only"
    elif rid == "exact_events": rp, rc = counts["exact_events"], "exact_link_only"
    elif rid == "exact_negatives": rp, rc = counts["exact_negatives"], "exact_link_only"
    elif rid == "excluded_rows": rp, rc = counts["excluded_rows"], "accepted_non_exact_excluded"
    elif rid == "excluded_events": rp, rc = counts["excluded_events"], "accepted_non_exact_excluded"
    elif rid == "full_ges_auprc":
        rp, rlo, rhi, rc = interval.loc["Full GES", ["point_auprc", "auprc_ci_lower", "auprc_ci_upper", "auprc_null_status"]]
    elif rid == "full_ges_auprc_minus_prevalence":
        z = valid_rep[f"{slug('Full GES')}_auprc"] - prev_exact
        rp, (rlo, rhi) = point.loc["Full GES", "auprc"] - prev_exact, ci(z)
        rc = "supported_above_zero" if rlo > 0 else "supported_below_zero" if rhi < 0 else "interval_includes_zero"
    elif rid == "full_ges_auroc":
        rp, rlo, rhi, rc = interval.loc["Full GES", ["point_auroc", "auroc_ci_lower", "auroc_ci_upper", "auroc_null_status"]]
    elif rid == "full_ges_auroc_minus_0_50":
        z = valid_rep[f"{slug('Full GES')}_auroc"] - .5
        rp, (rlo, rhi) = point.loc["Full GES", "auroc"] - .5, ci(z)
        rc = "supported_above_zero" if rlo > 0 else "supported_below_zero" if rhi < 0 else "interval_includes_zero"
    elif typ == "paired":
        q = pair_lookup.loc[(r["comparator"], r["metric"])]
        rp, rlo, rhi, rc = q[
            ["point_difference_full_minus_comparator", "difference_ci_lower", "difference_ci_upper", "paired_interval_status"]
        ]
    else:
        raise RuntimeError(f"Unhandled historical row: {rid}")

    tol = 0 if typ == "cohort" else 5.1e-7
    concordance_rows.append({
        **r,
        "reproduced_point": float(rp), "absolute_point_difference": abs(float(rp)-float(r["historical_point"])),
        "point_rounding_tolerance": tol,
        "point_reproduced_at_recorded_precision": abs(float(rp)-float(r["historical_point"])) <= tol,
        "reproduced_ci_lower": None if pd.isna(rlo) else float(rlo),
        "reproduced_ci_upper": None if pd.isna(rhi) else float(rhi),
        "reproduced_conclusion": str(rc),
        "scientific_conclusion_concordant": str(rc) == r["historical_conclusion"],
        "bootstrap_stream_note": "Historical row-level replicates were not serialized; independent PCG64 seed-42 intervals are stored separately.",
    })
concordance = pd.DataFrame(concordance_rows)


# ------------------------- QC -------------------------
checks = []
def check(name, passed, details):
    checks.append({"check_name": name, "passed": bool(passed), "details": native(details)})

check("source_hash", sha(SOURCE) == SOURCE_SHA, sha(SOURCE))
check("source_sidecar", sidecar_ok(SOURCE), str(SOURCE)+".sha256")
check("prior_manifest_hash", sha(PRIOR_MANIFEST) == PRIOR_MANIFEST_SHA, sha(PRIOR_MANIFEST))
check("prior_manifest_sidecar", sidecar_ok(PRIOR_MANIFEST), str(PRIOR_MANIFEST)+".sha256")
check("source_dimensions", df.shape == (EXPECTED["source_rows"], EXPECTED["source_cols"]), df.shape)
check("source_key", df[key].notna().all() and df[key].astype(str).nunique() == len(df), key)
check("source_accounting", int(y_all.sum()) == EXPECTED["source_events"] and int((1-y_all).sum()) == EXPECTED["source_negatives"], {})
check("score_columns", not missing, MODELS)
check("exact_mask_resolved", bool(exact_methods), exact_methods)
check("exact_accounting", all(counts[k] == EXPECTED[k] for k in counts), counts)
check("grouped_metric_validation", all(v["ap_abs_diff"] <= 1e-12 and v["auc_abs_diff"] <= 1e-12 for v in metric_validation.values()), metric_validation)
check("nine_point_estimates", len(points) == 9, len(points))
check("bootstrap_attempts", len(replicates) == N_BOOT, len(replicates))
check("bootstrap_valid", len(valid_rep) == N_BOOT, len(valid_rep))
check("bootstrap_invalid", invalid == 0, invalid)
check("nine_model_intervals", len(intervals) == 9, len(intervals))
check("sixteen_paired_rows", len(paired) == 16, len(paired))
check("secondary_holm_rows", len(holm_table) == 10, len(holm_table))
check("secondary_supported", holm_table["holm_supported_at_0_05"].all() and (holm_table["paired_interval_status"] == "full_ges_supported_higher").all(), holm_table.to_dict("records"))
check("historical_points", concordance["point_reproduced_at_recorded_precision"].all(), concordance[["result_id", "absolute_point_difference"]].to_dict("records"))
check("historical_conclusions", concordance["scientific_conclusion_concordant"].all(), concordance[["result_id", "historical_conclusion", "reproduced_conclusion"]].to_dict("records"))
check("frozen_sources_unchanged", sha(SOURCE) == SOURCE_SHA and sha(PRIOR_MANIFEST) == PRIOR_MANIFEST_SHA, {})

failed = [c for c in checks if not c["passed"]]
if failed: raise RuntimeError("QC failed before writing:\n" + json.dumps(native(failed), indent=2))


# ------------------------- write and read back tables -------------------------
write_csv(P["inventory"], inventory)
write_csv(P["points"], points)
write_parquet(P["replicates"], replicates)
write_csv(P["intervals"], intervals)
write_csv(P["paired"], paired)
write_csv(P["holm"], holm_table)
write_csv(P["historical"], historical)
write_csv(P["concordance"], concordance)

readback = {
    "inventory": len(pd.read_csv(P["inventory"])) == 3,
    "points": len(pd.read_csv(P["points"])) == 9,
    "replicates": len(pd.read_parquet(P["replicates"])) == N_BOOT,
    "intervals": len(pd.read_csv(P["intervals"])) == 9,
    "paired": len(pd.read_csv(P["paired"])) == 16,
    "holm": len(pd.read_csv(P["holm"])) == 10,
    "historical": len(pd.read_csv(P["historical"])) == len(historical),
    "concordance": len(pd.read_csv(P["concordance"])) == len(concordance),
}
if not all(readback.values()): raise RuntimeError(f"Table readback failed: {readback}")

qc_payload = {
    "cell_id": "6C-4G0", "package_version": "v1", "created_utc": CREATED_UTC,
    "analysis": "exact_link_only_sensitivity_materialization",
    "source": {"path": str(SOURCE), "sha256": sha(SOURCE)},
    "prior_manifest": {"path": str(PRIOR_MANIFEST), "sha256": sha(PRIOR_MANIFEST)},
    "exact_link_resolution_methods": exact_methods,
    "bootstrap": {
        "method": "paired nonparametric row bootstrap", "rng": "numpy.random.Generator",
        "bit_generator": type(rng.bit_generator).__name__, "seed": SEED,
        "attempts": N_BOOT, "valid": len(valid_rep), "invalid_one_class": invalid,
    },
    "checks": checks, "table_readback": readback,
    "passed_checks": sum(c["passed"] for c in checks), "failed_checks": sum(not c["passed"] for c in checks),
    "decision": "PASS_STAGE6C_EXACT_LINK_SENSITIVITY_RESULT_CATEGORY_MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED",
}
write_json(P["qc"], qc_payload)

for k in ["inventory", "points", "replicates", "intervals", "paired", "holm", "historical", "concordance", "qc"]:
    sidecar(P[k])
    if not sidecar_ok(P[k]): raise RuntimeError(f"Sidecar failed: {P[k]}")


def artifact(k):
    path = P[k]
    out = {
        "artifact_key": k, "path": str(path), "relative_path": str(path.relative_to(ROOT)),
        "sha256": sha(path), "bytes": path.stat().st_size,
        "sidecar_path": str(path.with_name(path.name+".sha256")), "sidecar_verified": sidecar_ok(path),
    }
    if path.suffix == ".csv":
        x = pd.read_csv(path); out.update(rows=len(x), columns=x.shape[1])
    elif path.suffix == ".parquet":
        m = pq.ParquetFile(path).metadata; out.update(rows=m.num_rows, columns=m.num_columns)
    else:
        json.loads(path.read_text()); out["json_readback"] = True
    return out


artifact_keys = ["inventory", "points", "replicates", "intervals", "paired", "holm", "historical", "concordance", "qc"]
manifest = {
    "cell_id": "6C-4G0", "package_version": "v1", "notebook_name": NOTEBOOK_NAME, "created_utc": CREATED_UTC,
    "authorized_category": "exact_link_only_sensitivity",
    "immutable_sources": {
        "stage6b_primary_evaluable": {"path": str(SOURCE), "sha256": sha(SOURCE), "expected_sha256": SOURCE_SHA},
        "prior_6c4f0_manifest": {"path": str(PRIOR_MANIFEST), "sha256": sha(PRIOR_MANIFEST), "expected_sha256": PRIOR_MANIFEST_SHA},
    },
    "analysis_lock": {
        "outcome": OUTCOME, "scores": MODELS, "exact_link_resolution_methods": exact_methods,
        "bootstrap_seed": SEED, "bootstrap_attempts": N_BOOT,
        "principal_comparators": PRINCIPAL, "secondary_comparators": SECONDARY,
        "secondary_multiplicity": "separate five-test Holm families for AUPRC and AUROC",
    },
    "cohort_accounting": inventory.to_dict("records"),
    "result_summary": {
        "full_ges_exact_link_auprc": float(point.loc["Full GES", "auprc"]),
        "full_ges_exact_link_auroc": float(point.loc["Full GES", "auroc"]),
        "historical_points_reproduced": bool(concordance["point_reproduced_at_recorded_precision"].all()),
        "historical_conclusions_concordant": bool(concordance["scientific_conclusion_concordant"].all()),
        "all_secondary_holm_comparisons_supported": bool(holm_table["holm_supported_at_0_05"].all()),
    },
    "software": {
        "python": sys.version, "platform": platform.platform(), "numpy": np.__version__,
        "pandas": pd.__version__, "pyarrow": pyarrow.__version__, "scikit_learn": sklearn.__version__,
    },
    "artifacts": [artifact(k) for k in artifact_keys],
    "scientific_boundary": {
        "frozen_inputs_modified": False, "scores_refit_or_recalibrated": False,
        "thresholds_or_weights_changed": False, "linkage_decisions_changed": False,
        "outcome_definition_changed": False, "experiment_2_started": False,
        "interpretation": (
            "Excluding 167 accepted non-exact links does not materially change the principal result. "
            "Full GES remains clearly above no-star GES, has a modest AUPRC but not AUROC advantage "
            "over review stars, and does not exceed combined metadata on primary AUPRC."
        ),
    },
    "decision": "PASS_STAGE6C_EXACT_LINK_SENSITIVITY_RESULT_CATEGORY_MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED",
    "next_authorized_category": "alternative_primary_and_secondary_evidence_drift_materialization",
}
manifest_hash = write_json(P["manifest"], manifest)
sidecar(P["manifest"])

# Final package and immutable-source reverification.
manifest_rb = json.loads(P["manifest"].read_text())
if not sidecar_ok(P["manifest"]): raise RuntimeError("Manifest sidecar failed.")
if manifest_rb["decision"] != manifest["decision"]: raise RuntimeError("Manifest decision mismatch.")
if manifest_rb["scientific_boundary"]["experiment_2_started"] is not False: raise RuntimeError("Experiment 2 boundary failed.")
for a in manifest_rb["artifacts"]:
    path = Path(a["path"])
    if sha(path) != a["sha256"] or not sidecar_ok(path): raise RuntimeError(f"Final artifact verification failed: {path}")
if sha(SOURCE) != SOURCE_SHA or sha(PRIOR_MANIFEST) != PRIOR_MANIFEST_SHA:
    raise RuntimeError("An immutable source changed during Cell 6C-4G0.")


# ------------------------- controlled output -------------------------
print("\n" + "="*160)
print("STAGE 6C STEP 4G — CELL 6C-4G0 — EXACT-LINK-ONLY SENSITIVITY RESULT CATEGORY")
print("="*160)
print(f"Stage 6B source hash                    : PASS ({sha(SOURCE)})")
print(f"Prior Cell 6C-4F0 manifest              : PASS ({sha(PRIOR_MANIFEST)})")
print(f"Exact-link resolution                   : PASS ({'; '.join(exact_methods)})")
print(f"Complete evaluable cohort               : {EXPECTED['source_rows']:,} rows | {EXPECTED['source_events']:,} events | {EXPECTED['source_negatives']:,} negatives")
print(f"Exact-link cohort                       : {counts['exact_rows']:,} rows | {counts['exact_events']:,} events | {counts['exact_negatives']:,} negatives")
print(f"Accepted non-exact excluded             : {counts['excluded_rows']:,} rows | {counts['excluded_events']:,} events | {counts['excluded_negatives']:,} negatives")
print(f"Bootstrap attempts                      : {N_BOOT:,}")
print(f"Valid / invalid replicates              : {len(valid_rep):,} / {invalid:,}")
print(f"Bootstrap elapsed                       : {elapsed/60:.2f} minutes")
print("Secondary Holm families                 : 5 AUPRC + 5 AUROC")
print(f"Historical point results                : PASS ({int(concordance.point_reproduced_at_recorded_precision.sum())}/{len(concordance)})")
print(f"Historical scientific conclusions       : PASS ({int(concordance.scientific_conclusion_concordant.sum())}/{len(concordance)})")
print(f"Fresh QC                                : PASS ({qc_payload['passed_checks']}/{len(checks)})")
for label, k in [
    ("Cohort inventory", "inventory"), ("Point estimates", "points"),
    ("Bootstrap replicates", "replicates"), ("Model intervals", "intervals"),
    ("Paired inference", "paired"), ("Secondary Holm table", "holm"),
    ("Historical results", "historical"), ("Concordance table", "concordance"),
    ("QC", "qc"), ("Manifest", "manifest")
]:
    print(f"{label:40s}: {P[k]}")
print(f"Manifest SHA-256                        : {manifest_hash}")

print("\nEXACT-LINK COHORT ACCOUNTING")
print(inventory.to_string(index=False))
print("\nREPRODUCED EXACT-LINK MODEL INTERVALS")
print(intervals.to_string(index=False))
print("\nREPRODUCED EXACT-LINK PAIRED INFERENCE")
print(paired.to_string(index=False))

print("\nCELL DECISION")
print("PASS_STAGE6C_EXACT_LINK_SENSITIVITY_RESULT_CATEGORY_MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED")
print(
    "The fifth of eight Stage 6C result categories is independently materialized. "
    "The next authorized category is alternative-primary and secondary evidence-drift materialization. "
    "Experiment 2 has not started."
)
print("="*160)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Use this Colab notebook file name: GES_Stage6C_Cell_6C_4G0_Exact_Link_Sensitivity_Materialization.ipynb

Preparing exact-link cohort: 66,469 rows, 6,433 events, 60,036 negatives
Exact grouped metric validation against scikit-learn: PASS
  Completed 250/2,000 replicates | valid 250
  Completed 500/2,000 replicates | valid 500
  Completed 750/2,000 replicates | valid 750
  Completed 1,000/2,000 replicates | valid 1,000
  Completed 1,250/2,000 replicates | valid 1,250
  Completed 1,500/2,000 replicates | valid 1,500
  Completed 1,750/2,000 replicates | valid 1,750
  Completed 2,000/2,000 replicates | valid 2,000

STAGE 6C STEP 4G — CELL 6C-4G0 — EXACT-LINK-ONLY SENSITIVITY RESULT CATEGORY
Stage 6B source hash                    : PASS (c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038)
Prior Cell 6C-4F0 manifest              : PASS (34646aa38862bb

In [4]:
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Use this Colab notebook file name: GES_Stage6C_Cell_6C_4G0_Exact_Link_Sensitivity_Materialization.ipynb

Preparing exact-link cohort: 66,469 rows, 6,433 events, 60,036 negatives
Exact grouped metric validation against scikit-learn: PASS
  Completed 250/2,000 replicates | valid 250
  Completed 500/2,000 replicates | valid 500
  Completed 750/2,000 replicates | valid 750
  Completed 1,000/2,000 replicates | valid 1,000
  Completed 1,250/2,000 replicates | valid 1,250
  Completed 1,500/2,000 replicates | valid 1,500
  Completed 1,750/2,000 replicates | valid 1,750
  Completed 2,000/2,000 replicates | valid 2,000

================================================================================================================================================================
STAGE 6C STEP 4G — CELL 6C-4G0 — EXACT-LINK-ONLY SENSITIVITY RESULT CATEGORY
================================================================================================================================================================
Stage 6B source hash                    : PASS (c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038)
Prior Cell 6C-4F0 manifest              : PASS (34646aa38862bbc425a7373a90e49c336023bc8e0e37ba82c8d9df5824a2e365)
Exact-link resolution                   : PASS (categorical:linkage_method; categorical:linkage_decision_category; rcv_equality:rcv_accession==t1_rcv_accession; rcv_equality:rcv_accession==linked_t1_rcv_accession)
Complete evaluable cohort               : 66,636 rows | 6,485 events | 60,151 negatives
Exact-link cohort                       : 66,469 rows | 6,433 events | 60,036 negatives
Accepted non-exact excluded             : 167 rows | 52 events | 115 negatives
Bootstrap attempts                      : 2,000
Valid / invalid replicates              : 2,000 / 0
Bootstrap elapsed                       : 0.52 minutes
Secondary Holm families                 : 5 AUPRC + 5 AUROC
Historical point results                : PASS (15/15)
Historical scientific conclusions       : PASS (15/15)
Fresh QC                                : PASS (22/22)
Cohort inventory                        : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/tables/stage6_temporal_validation/stage6c_4g0_exact_link_sensitivity_materialization_v1/stage6c_exact_link_cohort_inventory_v1.csv
Point estimates                         : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/tables/stage6_temporal_validation/stage6c_4g0_exact_link_sensitivity_materialization_v1/stage6c_exact_link_point_estimates_v1.csv
Bootstrap replicates                    : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/tables/stage6_temporal_validation/stage6c_4g0_exact_link_sensitivity_materialization_v1/stage6c_exact_link_bootstrap_replicates_v1.parquet
Model intervals                         : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/tables/stage6_temporal_validation/stage6c_4g0_exact_link_sensitivity_materialization_v1/stage6c_exact_link_model_bootstrap_intervals_v1.csv
Paired inference                        : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/tables/stage6_temporal_validation/stage6c_4g0_exact_link_sensitivity_materialization_v1/stage6c_exact_link_paired_inference_v1.csv
Secondary Holm table                    : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/tables/stage6_temporal_validation/stage6c_4g0_exact_link_sensitivity_materialization_v1/stage6c_exact_link_secondary_holm_v1.csv
Historical results                      : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/tables/stage6_temporal_validation/stage6c_4g0_exact_link_sensitivity_materialization_v1/stage6c_exact_link_historical_results_v1.csv
Concordance table                       : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/tables/stage6_temporal_validation/stage6c_4g0_exact_link_sensitivity_materialization_v1/stage6c_exact_link_historical_vs_reproduced_concordance_v1.csv
QC                                      : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/quality_checks/stage6_temporal_validation/stage6c_4g0_exact_link_sensitivity_materialization_v1/stage6c_4g0_exact_link_sensitivity_qc_v1.json
Manifest                                : /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage6_temporal_validation/stage6c_4g0_exact_link_sensitivity_materialization_v1/stage6c_4g0_exact_link_sensitivity_manifest_v1.json
Manifest SHA-256                        : f9586e0609f76f9adda52e153dfcb7800e77b0840a1eb076c44936777ba8611f

EXACT-LINK COHORT ACCOUNTING
             linkage_subset  rows  events  negatives  event_prevalence  included_in_exact_link_sensitivity
 complete_primary_evaluable 66636    6485      60151          0.097320                               False
            exact_link_only 66469    6433      60036          0.096782                                True
accepted_non_exact_excluded   167      52        115          0.311377                               False

REPRODUCED EXACT-LINK MODEL INTERVALS
 linkage_subset                  model  point_auprc  auprc_ci_lower  auprc_ci_upper  auprc_null                       auprc_null_status  point_auroc  auroc_ci_lower  auroc_ci_upper  auroc_null    auroc_null_status  valid_bootstrap_replicates  invalid_one_class_replicates
exact_link_only               Full GES     0.112073        0.107727        0.116975    0.096782   supported_above_exact_link_prevalence     0.535940        0.528691        0.542607         0.5 supported_above_0_50                        2000                             0
exact_link_only            No-star GES     0.096158        0.092313        0.100781    0.096782 interval_includes_exact_link_prevalence     0.448152        0.440340        0.455285         0.5 supported_below_0_50                        2000                             0
exact_link_only           Review stars     0.107685        0.104789        0.110620    0.096782   supported_above_exact_link_prevalence     0.532877        0.528005        0.537803         0.5 supported_above_0_50                        2000                             0
exact_link_only      Combined metadata     0.113128        0.108700        0.118137    0.096782   supported_above_exact_link_prevalence     0.532699        0.525246        0.539447         0.5 supported_above_0_50                        2000                             0
exact_link_only               Conflict     0.101572        0.098873        0.104262    0.096782   supported_above_exact_link_prevalence     0.513223        0.510567        0.515773         0.5 supported_above_0_50                        2000                             0
exact_link_only                Recency     0.082346        0.079902        0.084903    0.096782   supported_below_exact_link_prevalence     0.438046        0.430491        0.445146         0.5 supported_below_0_50                        2000                             0
exact_link_only      Submitter support     0.091770        0.089519        0.094101    0.096782   supported_below_exact_link_prevalence     0.470269        0.464686        0.475751         0.5 supported_below_0_50                        2000                             0
exact_link_only Classification entropy     0.105714        0.102692        0.108854    0.096782   supported_above_exact_link_prevalence     0.518658        0.515193        0.522171         0.5 supported_above_0_50                        2000                             0
exact_link_only          Additive risk     0.098953        0.095942        0.102030    0.096782 interval_includes_exact_link_prevalence     0.482594        0.476772        0.488617         0.5 supported_below_0_50                        2000                             0

REPRODUCED EXACT-LINK PAIRED INFERENCE
 linkage_subset comparator_family metric             comparator  point_difference_full_minus_comparator  difference_ci_lower  difference_ci_upper    paired_interval_status  bootstrap_sign_p_value  valid_bootstrap_replicates  holm_adjusted_bootstrap_sign_p holm_supported_at_0_05
exact_link_only         principal  AUPRC            No-star GES                                0.015915             0.014591             0.017252 full_ges_supported_higher                0.001000                        2000                             NaN                    NaN
exact_link_only         principal  AUROC            No-star GES                                0.087788             0.083232             0.092255 full_ges_supported_higher                0.001000                        2000                             NaN                    NaN
exact_link_only         principal  AUPRC           Review stars                                0.004388             0.000977             0.008276 full_ges_supported_higher                0.011994                        2000                             NaN                    NaN
exact_link_only         principal  AUROC           Review stars                                0.003064            -0.001854             0.007526    interval_includes_zero                0.224888                        2000                             NaN                    NaN
exact_link_only         principal  AUPRC      Combined metadata                               -0.001055            -0.002298             0.000229    interval_includes_zero                0.107946                        2000                             NaN                    NaN
exact_link_only         principal  AUROC      Combined metadata                                0.003241             0.001495             0.005108 full_ges_supported_higher                0.001000                        2000                             NaN                    NaN
exact_link_only         secondary  AUPRC               Conflict                                0.010501             0.007972             0.013677 full_ges_supported_higher                0.001000                        2000                        0.004998                   True
exact_link_only         secondary  AUROC               Conflict                                0.022717             0.015828             0.029081 full_ges_supported_higher                0.001000                        2000                        0.004998                   True
exact_link_only         secondary  AUPRC                Recency                                0.029727             0.026599             0.033435 full_ges_supported_higher                0.001000                        2000                        0.004998                   True
exact_link_only         secondary  AUROC                Recency                                0.097894             0.092246             0.103603 full_ges_supported_higher                0.001000                        2000                        0.004998                   True
exact_link_only         secondary  AUPRC      Submitter support                                0.020303             0.016615             0.024685 full_ges_supported_higher                0.001000                        2000                        0.004998                   True
exact_link_only         secondary  AUROC      Submitter support                                0.065671             0.058298             0.072711 full_ges_supported_higher                0.001000                        2000                        0.004998                   True
exact_link_only         secondary  AUPRC Classification entropy                                0.006359             0.003800             0.009455 full_ges_supported_higher                0.001000                        2000                        0.004998                   True
exact_link_only         secondary  AUROC Classification entropy                                0.017282             0.010441             0.023656 full_ges_supported_higher                0.001000                        2000                        0.004998                   True
exact_link_only         secondary  AUPRC          Additive risk                                0.013120             0.009985             0.016520 full_ges_supported_higher                0.001000                        2000                        0.004998                   True
exact_link_only         secondary  AUROC          Additive risk                                0.053347             0.048388             0.058046 full_ges_supported_higher                0.001000                        2000                        0.004998                   True

CELL DECISION
PASS_STAGE6C_EXACT_LINK_SENSITIVITY_RESULT_CATEGORY_MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED
The fifth of eight Stage 6C result categories is independently materialized. The next authorized category is alternative-primary and secondary evidence-drift materialization. Experiment 2 has not started.
================================================================================================================================================================

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 45)